In [35]:
%run ./combinations_of_permutations.ipynb
from itertools import combinations, product
import numpy as np
from collections import Counter
import datetime
import re

What is our plan here for the six term constant cover search? 
We should generate all possible length tuples that we can search over.
Then it would be nice to write a general algorithm which solves all cases.
Peter's algorithm was for very specific cases if we remember correctly.


1. Start search from the shorter permutations (3,3,3,3,2,2) etc up to (18^6).
2. Record searched tuples to a file so we do not have to research.
3. Record found constant covers to another file.
4. We can use the degree sequence of the polynomials to rule things out more quickly.
5. We generate first row, check if possible, if not reject, if yes add second row, repeat until success or rejection

In [2]:
# first m rows of the above based on a k-permutation of [n]
def genPermMatrixByRow(p,n,m=None,k=None):
    if m==None:
        m=n
    l=len(p)
    if k==None:
        k=l
    if l==n:
        return matrix(QQ,m,n,lambda x,y:p[x]==y)
    return QQ(factorial(n-k))/QQ(binomial(n,k))*matrix(QQ,m,n,lambda x,y:sum([binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(l)]))


#Input: A k permutation p, an integer n to stretch it to, starting from start and going up to end non-inclusive.
#Output: The rows of the fuzzy permutation matrix between start and end non-inclusive.
def genPermMatrixBetweenRows(p,n,start=None,end=None,k=None):
    if (start > end):
        raise ValueError
    if start == None:
        start = 0
    if end == None:
        end = n
    l = len(p)
    if k == None:
        k = l
    
    rows = []
    for rowx in range(start,end):
        rows.append(QQ(factorial(n-k))/QQ(binomial(n,k))*vector(QQ,n, [sum([binomial(rowx,j)*binomial(n-1-rowx,k-1-j)*binomial(columny,p[j])*binomial(n-1-columny,k-1-p[j]) for j in range(l)]) for columny in range(n)]))                      
    return matrix(QQ,rows)

R.<x,y> = PolynomialRing(QQ)
# polynomial of first m rows based on a k-permutation of [n]
def genPermPolynomialByRow(p,n,m=None,k=None):
    if m==None:
        m=n
    l=len(p)
    if k==None:
        k=l
    return factorial(n-k)/binomial(n,k)*sum([binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(l)])

#Input: Two positive integers n,r representing the length of the longest permutation and the number of permutations.
#Output: A list of possible length sequences for a constant cover according to lemma 4.19 of Six Permutation Patterns Force Quasirandomness.
def possibleLengths(n,r):
    stack = []
    stack.append([n,n])
    while len(stack) > 0:
        lenList = stack.pop()
        if (len(lenList) >= r):
            if (lenList.count(2) < 3):
                yield lenList
            continue
        lower_bound = lemma419(lenList,n,r)
        for k in range(lower_bound,lenList[-1]+1):
            stack.append(lenList+[k])
    return None

#Input: A list of lengths, the maximum length n, and the maximum number of terms r.
#Output: The lower bound on the next term in the sequence according to lemma 4.19 of Six Permutation Patterns Force Quasirandomness
def lemma419(lenList,n,r):
    if len(lenList) >= r:
        return None
    else:
        lower_bound = n+1 - sum([n-k+1 for k in lenList])
        return max(lower_bound,2)
    
#Input: A list of lengths
#Ouput: The upper bound on n as given by lemma 4.17 in Six Permutation Patterns Force Quasirandomness
def lemma417(lenList,n):
    r = len(lenList)
    upperbound = (n+1)*(r-1) - sum(lenList[0:-1]) + 2
    return upperbound

#Input: A domain size n and codomain size m with mleq n:
#Output: Yields an injection from domain to codomain given as a partial permutation
def partialPermutations(n,m):
    if n < m:
        raise ValueError   
    for unordered in combinations(range(n),m):
        for ordered in Permutations(unordered):
            yield ordered
    return None       

#Input: A partial permutation, extend by 1 or by 2, and the max allowed length.
#Output: A list of partial permutations extended by 1 or 2.
def extendPartial(partial, extend_by,max_length):
        #if the partial is already at max_length, it can be extended no further.
    if len(partial) == max_length:
        yield partial
        
        #if we are extending by more than 2, we cannot handle that currently.
    elif extend_by not in [1,2]:
        raise ValueError
        
        #if we are extending by 1 or we only have room for 1, perform extension
    elif (extend_by == 1) or (max_length - len(partial) == 1):
        for perm in [list(partial)+[i] for i in set(range(max_length)).difference(partial)]:
            yield perm 
        #otherwise we are extending by exactly 2, so perform the extension.
    else:
        for perm in [list(partial)+[i]+[j] for i in set(range(max_length)).difference(partial) for j in set(range(max_length)).difference(list(partial)+[i])]:
            yield perm
    return None

#Input: A permutation perm and an integer n at least as large as the length of the permutation. A polynomial ring
#Output: A rational polynomial in x and y which represents the fuzzy permutation matrix of perm.
def fuzzyPolynomial(perm,n, k, Ring):
    x, y = Ring.gens()
    p = j_perm(perm)
    f = factorial(n-k)/binomial(n,k)*sum(binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(k))
    return Ring(f)

#Input: A tuple of lengths to search for covers on
#Output: A list of strings containing either the found covers or a message of how they were ruled out.
def searchRecur(in_tuple):
        #acquire a copy of the length tuple
    len_tuple = list(in_tuple)  
    print(datetime.datetime.now())
        #set starting maximum rows. Manually change this parameter if you like
    max_rows = 1
    
        #find n
    n = len_tuple[0]
    
        #set up list to return covers into
    found_covers = []
            #a list containing
    search_data = []
        #is search_data is None no data will be collected.
    #search_data = None
    
        #check if we can rule out the tuple without searching it.
    tuple_ruled_out = checkTuple(len_tuple)
    if tuple_ruled_out:
            #if we have ruled it out, return a message of how.
        return ([tuple_ruled_out],None)

    
        #perform search on tuple
    searchHelper([],max_rows, n,found_covers,len_tuple,search_data)
        #if we did not find any, document that
    if len(found_covers) == 0:
        return ([f"{len_tuple}: No covers found on full search."],search_data)
    
    found_inf_families = []
        #get rid of fractional coefficients.
    for cover in found_covers:
        if not type(cover[0]) == str:
            cover[0] = [coeff/gcd(cover[0]) for coeff in cover[0]]
        else:
            found_inf_families.append(cover)
        
        #if we found covers, reduce them by D4 symmetries.
    found_covers_cc = [combination_of_permutations(cover[0],cover[1]) for cover in found_covers if not type(cover[0]) == str] 
    found_covers_cc = reduceByD4(found_covers_cc)
    
        #return covers found
    return ([str(cover) for cover in found_covers_cc + found_inf_families],search_data)
    
#A recursive function which performs a full search for constant covers on the given length tuple.
#Covers which are found are recorded to found_covers. 
#max_rows is the length of the partial permutations we are using for our search. n is the length of the longest permutation in len_tuple
#perm_array is a list of pairs. Those pairs are the partial permutations we are searching along with their vectorized matrix forms. 
def searchHelper(in_array, max_rows, n,found_covers, len_tuple, search_data):
        #make a copy of in_array for this scope.
    perm_array = list(in_array)
    
        #in_array is initially empty. In searchHelper we fill out our array until it has len(len_tuple) many entries. Then we check for cover potential.
    current_level = len(perm_array)
    #print(f"\r Current level:{current_level}",end="")
    
        #if we are at the end of the tuple, attempt to find a cover
    if current_level == len(len_tuple):
            #find the row vectors which are flattened fuzzy matrices
        rows = [vector(QQ, list(perm_array[i][1])) for i in range(current_level)]
            #build our matrix. We transpose it so that our fuzzy matrices are the columns
        A = matrix(QQ, rows).transpose()
            #Augment the matrix with the all 1's vector to see if there are constant covers
        augRank = A.augment(vector(QQ, [1]*(n*max_rows))).rank()
            #We use the system rank theorem. If rank(A) == rank(Aaug) and rank(A) = number of columns of (A), then the solution exists and is unique.
        if A.rank() == augRank and (max_rows < n) and (A.rank() == A.ncols()):
            uniqueSoln = A.solve_right(vector(QQ, [1]*(n*max_rows)))
                #We find the unique soln. If it contains a zero then we reject, otherwise we pass the unique solution to the future search.
            if 0 in uniqueSoln:
                updateSearchData(perm_array,search_data)
                return
            else:
                #print("passing unique soln at level 1")
                searchExtension(perm_array, max_rows+1,n,found_covers,len_tuple,0,A,uniqueSoln,search_data)
                #print("finish level 1 passing")
            #if rank(A) < rank(Aaug), then the system is inconsistent and we reject.
        elif A.rank() < augRank:
            updateSearchData(perm_array,search_data)
            return
        else:
            #the system has solutions, so we search its extensions.
            searchExtension(perm_array, max_rows+1,n,found_covers,len_tuple,0,A,[],search_data)
       
        #here our tuple is not yet full, so we continue to fill it.
    else:
            #loop over all possible partial permutations
        for parperm in partialPermutations(len_tuple[current_level],max_rows):
                #if we are not at level zero, and the current and previous elements of the length tuple are equal, then if this partial permutation violates our ordering, we do not search it.
            if (current_level != 0) and (len_tuple[current_level-1] == len_tuple[current_level]) and (str(parperm) < str(perm_array[-1][0])):
                continue
                #if the first entry of the first permutation is greater than (n+1)/2, then by D_4 symmetries we don't need to check it.
            elif (current_level == 0) and (2*(parperm[0]+1) > n+ 1):
                continue
            else:
                    #print which leading partial permutation we are currently working on.
                if current_level == 0:
                    #pass
                    print(f"\rExamining {parperm}")
                    
                    #take the perm_array we already had, and add the new partial permutation to its end.
                extended_array = perm_array + [[parperm, vector(QQ,genPermMatrixByRow(parperm,n,max_rows,len_tuple[current_level]).list())]]
                    #run searchHelper again with the extended array.
                searchHelper(extended_array,max_rows, n, found_covers, len_tuple, search_data)
    
    return
#A recursive function which accepts a filled out partial permutation array as input. This function extends the partials 
# and attempts to determine if there is a constant cover on its extensions.
def searchExtension(in_array, max_rows, n, found_covers, len_tuple, current_level,current_matrix,uniqueSoln,search_data):
    extend_by = 1
    next_max_rows = min([max_rows+extend_by,n])
    perm_array = list(in_array)
    #print(f"\r Extended Current level:{current_level}",end="")
    if current_level == len(len_tuple):
        matrix_extension = matrix(QQ,[vector(QQ, list(perm_array[i][1])) for i in range(current_level)]).transpose()
        A = current_matrix.stack(matrix_extension)
        
        if len(uniqueSoln) > 0:
            if (A*vector(QQ,uniqueSoln) == vector(QQ, [1]*(n*max_rows))):
                if (max_rows == n):
                    found_covers.append([list(uniqueSoln),[perm_array[i][0] for i in range(len(perm_array))]])
                    updateSearchData(perm_array,search_data)
                    print(f"Found cover with perms: {[perm_array[i][0] for i in range(len(perm_array))]}")
                else:
                    searchExtension(perm_array, next_max_rows,n,found_covers,len_tuple,0,A,uniqueSoln,search_data) 
            return
            
        augRank = A.augment(vector(QQ, [1]*(n*max_rows))).rank()
        if A.rank() == augRank and (max_rows < n) and (A.rank() == A.ncols()):
            uniqueSoln = A.solve_right(vector(QQ, [1]*(n*max_rows)))
            if 0 in uniqueSoln:
                updateSearchData(perm_array,search_data)
                return
            else:
                #print(f"passing level {max_rows} solution")
                searchExtension(perm_array, next_max_rows,n,found_covers,len_tuple,0,A,uniqueSoln,search_data) 
                #print(f"finish passing {max_rows} solution")
        elif A.rank() < augRank:
            updateSearchData(perm_array,search_data)
            return
        elif (max_rows == n):
            if A.rank() == A.ncols():
                soln = A.solve_right(vector(QQ, [1]*(n*max_rows)))  
                if 0 not in soln:
                    found_covers.append([list(soln),[perm_array[i][0] for i in range(len(perm_array))]])
                    print(f"Found cover with perms: {[perm_array[i][0] for i in range(len(perm_array))]}")
                    updateSearchData(perm_array,search_data)
                else:
                    updateSearchData(perm_array,search_data)
                    return
            else:
                    #We have an infinite family of covers in this case.
                found_covers.append(infiniteFamilyBasis([perm_array[i][0] for i in range(len(perm_array))], A))
                print(f"Found infFam with perms: {[perm_array[i][0] for i in range(len(perm_array))]}")
                updateSearchData(perm_array,search_data)
                #found_covers.append(["Infinite family:"]+[perm_array[i][0] for i in range(len(perm_array))])
        else:
            searchExtension(perm_array, next_max_rows,n,found_covers,len_tuple,0,A,uniqueSoln,search_data) 
    else:
        for extendedPartial in extendPartial(perm_array[current_level][0],max_rows - len(perm_array[current_level][0]),len_tuple[current_level]):
            if (current_level > 0) and (len_tuple[current_level-1] == len_tuple[current_level]) and ((str(extendedPartial) < str(perm_array[current_level-1][0])) or (len(extendedPartial) == len_tuple[current_level] and (str(extendedPartial) == str(perm_array[current_level-1][0])))):    
                continue #if this does not meet our ordering and uniqueness conditions, we reject early.
 #           elif (current_level+1 == len(len_tuple)):
 #               for perm in perm_array:
 #                   if perm[0] == extendedPartial:
 #                       continue
            start = current_matrix.nrows()//n
            extended_array = perm_array[:current_level] + [[extendedPartial,vector(QQ,genPermMatrixBetweenRows(extendedPartial,n,start,max_rows,len_tuple[current_level]).list())]] + perm_array[current_level+1:]
            searchExtension(extended_array, max_rows,n,found_covers,len_tuple,current_level+1,current_matrix,uniqueSoln,search_data)
    return

def checkTuple(len_tuple):
    if not checkFirstRowFillable(len_tuple):
        return f"Covers of type {len_tuple} cannot fill the first row with enough non-zero entries."
    #if (len(len_tuple) == 6) and (len_tuple[-2]==2) and (len_tuple[0] >= 6):
    #    return f"{len_tuple} covers do not exist since {len_tuple[:-1]} covers do not exist."
    #if all((length == len_tuple[0] for length in len_tuple)):
    #    if len_tuple[0] == len(len_tuple):
    #        return f"Covers of type {len_tuple} are the latin squares of order {len_tuple[0]}."
    #    else: 
    #        return f"Covers of type {len_tuple} are ruled out by length tuple alone."
    elif len_tuple[0] < 7:
        repeated = checkRepeatedEntries(len_tuple)
        if repeated:
            if repeated == "singles_equality" or repeated == "pair_equality":
                pass
                #return f"Covers of type {len_tuple} have equality for repeated entry bound. Perform an additional check with simplified search space."
            else:
                return f"Covers of type {len_tuple} are ruled out by repeated entries"
        
    return False
    
#Each permutation of length k has n-k+1 non-zero entries in the first row. If the sum of these is less than n, we cannot have a constant cover.
#Input: len_tuple
#Output: Returns false is first row is not fillable with non-zero entries. Returns true otherwise.
def checkFirstRowFillable(len_tuple):
    n = len_tuple[0]
    if sum([n - k +1 for k in len_tuple]) < n:
        return False
    return True


def checkRepeatedEntries(len_tuple):
    n = len_tuple[0]
    min_zeroes = n**2 - sum([((n-k+1)**2)*k for k in len_tuple[:-1]])
    try:
        most_repeats_single = most_repeated_entry_count[(n,len_tuple[-1])][0]
    except KeyError:
        repeated_entries = findRepeatedEntries(n,len_tuple[-1])
        most_repeated_entry_count.update({(n,len_tuple[-1]) : repeated_entries})
        most_repeats_single = most_repeated_entry_count[(n,len_tuple[-1])][0]
        
    if min_zeroes > most_repeats_single:
        return True
    
    if min_zeroes == most_repeats_single:
        return "singles_equality"
        
    min_zeroes = n**2 - sum([((n-k+1)**2)*k for k in len_tuple[:-2]])
    try:
        most_repeats_pair = most_repeated_entry_count[(n,len_tuple[-2],len_tuple[-1])][0]
    except KeyError:
        repeated_entries = findRepeatedEntriesPair(n,len_tuple[-2],len_tuple[-1])
        most_repeated_entry_count.update({(n,len_tuple[-2], len_tuple[-1]) : repeated_entries})   
        most_repeats_pair = most_repeated_entry_count[(n,len_tuple[-2],len_tuple[-1])][0]
        
    if min_zeroes > most_repeats_pair:
        return True
    
    if min_zeroes == most_repeats_pair:
        return "pair_equality"
    
    return False

#Computes the most repeated entries for permutation matrices for a given n
def findRepeatedEntries(n,k):
    current_max = 0
    max_achievers = []
    for perm in Permutations(k):
        counts = Counter([entry for entry in genPermMatrix(j_perm(perm),n).list() if entry != 0])
        most_reps = counts.most_common(1)[0][1]
        
        if most_reps == current_max:
            max_achievers.append(perm)
        if most_reps > current_max:
            current_max = most_reps
            max_achievers = [perm]
    return (current_max, max_achievers)
        
#Computes the most repeated entires for pairs of permutation matrices with lengths up to n.
def findRepeatedEntriesPair(n,k1,k2):
    current_max = 0
    max_achievers = []
    for perm1 in Permutations(k1):
        u = genPermMatrixByRow(j_perm(perm1),n,n,k1).list()
        for perm2 in Permutations(k2):
            v = genPermMatrixByRow(j_perm(perm2),n,n,k2).list()
            coeffs=[leadingOne([v[j]-v[i],u[i]-u[j]]) for i in range(len(u)) for j in range(i)]
            for c in set(coeffs):
                this_comb=c[0]*vector(u)+c[1]*vector(v)
                w = this_comb.list()
                m = max([0]+[w.count(x) for x in w if (x!=0)])
                if m == current_max:
                    max_achievers.append((perm1,perm2))
                if m > current_max:
                    current_max = m
                    max_achievers = [(perm1,perm2)]
    return (current_max, max_achievers)
                    
def leadingOne(w):
    for x in w:
        if x!=0:
            scaled=tuple((1/x)*vector(w))
            return scaled
    return tuple(w)

#Input: A list of permutations which we have verified to be an infinite family of and a matrix whose columns are 
# the vectorized fuzzy permutation matrices.
#Output: A string representing the infinite family.
def infiniteFamilyBasis(family, familyMatrix):
    variables = ["t","s","r","q"]
    n = len(family[0]) 
    nullBasis = familyMatrix.right_kernel().basis()
    particularSoln = combination_of_permutations(list(familyMatrix.solve_right(vector(QQ,[1]*n**2))),family)
    nullCovers = [combination_of_permutations(list(basisVector),family) for basisVector in nullBasis]
    returnString = str(particularSoln)
    for i, basisVector in enumerate(nullBasis):
        returnString += "+" +str(variables[i]) + "*(" + str(combination_of_permutations(list(basisVector),family))[:-1] +")"
    return "infFam:" + returnString

#Input: A list of bases for infinite familes, a new infinite family basis
#ouput: If new infinite family basis is D4 equivalent to one in the old list, return True.
def d4EquivInfFams(basesList, newBasis):
    for basis in basesList:
        if len(basis) == len(newBasis):
            basisCoeffs = [sorted(term.get_coefficients()) for term in basis]
            newBasisCoeffs = [sorted(term.get_coefficients()) for term in newBasis]
            coeffComp = [basisCoeffs[i] == newBasisCoeffs[i] for i in range(len(basis))]
            if not all(coeffComp):
                return False
            basesListSymmetries = [rho.d4Symmetries(False) for rho in basis]
            inList = [newBasis[i] in basesListSymmetries[i] for i in range(len(basis))]
            if not all(inList):
                return False
            
            for i in range(8):
                symms = [symm[i] for symm in basesListSymmetries]
                if all([symms[i] == newBasis[i] for i in range(len(symms))]):
                    return True
    return False

#Input: A string, and if that string is old or new.
#Ouput: A list of combination_of_permutations which form a basis.
def textToBasis(age, string):
    basis = []
    if age == "new":
        string = string[7:]
        result = re.split(r'\+(?:t|s|r|q)\*',string)
        clean_result = [item for item in result if item]
        #print(clean_result)
        for string in clean_result:
            coeff_list = []
            term_list = []
            for i, char in enumerate(string):
                if char == '*':
                    coeff = int(string[i-1])
                    if i > 2 and string[i-2] == '-':
                        coeff *= -1
                    coeff_list.append(coeff)
                    term_string = string[i+2:].split(')')[0]
                    termutation =[]
                    for termchar in term_string:
                        termutation.append(int(termchar))
                    term_list.append(termutation)
            basis.append(combination_of_permutations(coeff_list,term_list))
    if age == "old":
        pass
    
    return basis

#Input
#Usage: updates the search_data
def updateSearchData(perm_array,search_data):
    if search_data is not None:
        partial_perm_list = [perm[0] for perm in perm_array]
        search_data.append([len(partial_perm_list[0]),partial_perm_list])
        if len(search_data)%10 == 0:
            print(f"The {len(search_data)}th case: {partial_perm_list}")
    return None

In [22]:
#data_array, has tuple, number of entries, number of d4 reduced inf families.
data_array =[]
working_tuple = -1
currentInfFams = []
with open("n5covers6", "r") as data_file:
    for line in data_file:
        line = line.strip()
        if line[0] == '[' and line[1] != "'":
            working_tuple += 1
            for inffam in currentInfFams:
                for term in inffam: 
                    print(term)
                print()
            currentInfFams = []
            data_array.append([line,0,0])
            print(line)
            if line[1] == 'e':
                for thing in data_array:
                    print(thing)
        elif line[0] == '[' and line[1] == "'":
            terms = line[:-1].split(", [")[1:]
            terms = [term[:-1] for term in terms]
            terms = [term.split(",") for term in terms]
            terms = [[int(entry) for entry in term] for term in terms]
            terms = [j_perm(term) for term in terms]
            n = len(terms[0])
            M = Matrix(QQ, [genPermMatrix(term, n).list() for term in terms]).transpose()
            line = infiniteFamilyBasis(terms, M)
            basis = textToBasis("new", line)
            lenSet = set()
            for term in basis:
                for term1 in term.get_terms():
                    lenSet.add(len(term1))
            lenTuple = [int(entry) for entry in data_array[working_tuple][0][1:-1].split(",")]
            good = True
            for i in lenTuple:
                if i not in lenSet:
                    good = False
            if not good:
                continue
            if currentInfFams == []:
                currentInfFams.append(basis)
                data_array[working_tuple][2] += 1
            if not d4EquivInfFams(currentInfFams, basis):
                currentInfFams.append(basis)
                data_array[working_tuple][2] += 1
                
        elif line[0] == 'i':
            basis = textToBasis("new", line)
            lenSet = set()
            for term in basis:
                for term1 in term.get_terms():
                    lenSet.add(len(term1))
            lenTuple = [int(entry) for entry in data_array[working_tuple][0][1:-1].split(",")]
            good = True
            for i in lenTuple:
                if i not in lenSet:
                    good = False
            if not good:
                continue
            if currentInfFams == []:
                currentInfFams.append(basis)
                data_array[working_tuple][2] += 1
            if not d4EquivInfFams(currentInfFams, basis):
                currentInfFams.append(basis)
                data_array[working_tuple][2] += 1 
        else:
            data_array[working_tuple][1] += 1
            
          
                
        
        #print(line)

[5, 5, 5, 5, 5, 2]
[5, 5, 5, 5, 5, 4]:
[5, 5, 5, 5, 4, 4]
[5, 5, 5, 5, 4, 3]
1*(12345) -1*(52341) + 2*(2143) + 8*(321) 
1*(12345) -1*(12435) -1*(52341) + 1*(52431) 

1*(12345) + 1*(52341) + 2*(3412) + 8*(123) 
1*(12345) -1*(12435) -1*(52341) + 1*(52431) 

1*(12345) -1*(52341) + 2*(2143) + 8*(321) 
1*(12345) -1*(13425) -1*(52341) + 1*(53421) 

1*(12435) -1*(52431) + 2*(2143) + 8*(321) 
1*(12435) -1*(13245) -1*(52431) + 1*(53241) 

1*(12435) -1*(52431) + 2*(2143) + 8*(321) 
1*(12435) -1*(13425) -1*(52431) + 1*(53421) 

1*(12345) + 1*(52341) + 2*(3412) + 8*(123) 
1*(12345) -1*(13245) -1*(52341) + 1*(53241) 

1*(12345) + 1*(52341) + 2*(3412) + 8*(123) 
1*(12345) -1*(13425) -1*(52341) + 1*(53421) 

1*(12435) + 1*(52431) + 2*(3412) + 8*(123) 
1*(12435) -1*(13245) -1*(52431) + 1*(53241) 

1*(12435) + 1*(52431) + 2*(3412) + 8*(123) 
1*(12435) -1*(13425) -1*(52431) + 1*(53421) 

1*(12345) -1*(52341) + 2*(2143) + 8*(321) 
1*(12345) -1*(14235) -1*(52341) + 1*(54231) 

1*(12345) -1*(52341) + 2*(21


8*(12) + 8*(21) 
1*(13245) -1*(13542) -1*(34215) + 1*(34512) 

8*(12) + 8*(21) 
1*(13254) -1*(13524) -1*(34251) + 1*(34521) 

8*(12) + 8*(21) 
1*(13425) -1*(13452) -1*(34125) + 1*(34152) 

8*(12) + 8*(21) 
1*(13245) -1*(13542) -1*(43215) + 1*(43512) 

8*(12) + 8*(21) 
1*(13254) -1*(13524) -1*(43251) + 1*(43521) 

8*(12) + 8*(21) 
1*(13425) -1*(13452) -1*(43125) + 1*(43152) 

8*(12) + 8*(21) 
1*(13245) -1*(13425) -1*(35241) + 1*(35421) 

8*(12) + 8*(21) 
1*(13254) -1*(13452) -1*(35214) + 1*(35412) 

8*(12) + 8*(21) 
1*(13524) -1*(13542) -1*(35124) + 1*(35142) 

8*(12) + 8*(21) 
1*(13245) -1*(13425) -1*(53241) + 1*(53421) 

8*(12) + 8*(21) 
1*(13254) -1*(13452) -1*(53214) + 1*(53412) 

8*(12) + 8*(21) 
1*(13524) -1*(13542) -1*(53124) + 1*(53142) 

8*(12) + 8*(21) 
1*(13245) -1*(14235) -1*(23145) + 1*(24135) 

8*(12) + 8*(21) 
1*(13245) -1*(14253) -1*(23145) + 1*(24153) 

8*(12) + 8*(21) 
1*(13254) -1*(14235) -1*(23154) + 1*(24135) 

8*(12) + 8*(21) 
1*(13254) -1*(14253) -1*(23154) + 1*(

1*(15243) -1*(15423) -1*(53241) + 1*(53421) 

8*(12) + 8*(21) 
1*(15324) -1*(15342) -1*(53124) + 1*(53142) 

8*(12) + 8*(21) 
1*(15234) -1*(15324) -1*(45231) + 1*(45321) 

8*(12) + 8*(21) 
1*(15243) -1*(15342) -1*(45213) + 1*(45312) 

8*(12) + 8*(21) 
1*(15423) -1*(15432) -1*(45123) + 1*(45132) 

8*(12) + 8*(21) 
1*(15234) -1*(15324) -1*(54231) + 1*(54321) 

8*(12) + 8*(21) 
1*(15243) -1*(15342) -1*(54213) + 1*(54312) 

8*(12) + 8*(21) 
1*(15423) -1*(15432) -1*(54123) + 1*(54132) 

8*(12) + 8*(21) 
1*(51234) -1*(51243) -1*(52134) + 1*(52143) 

8*(12) + 8*(21) 
1*(51324) -1*(51423) -1*(52314) + 1*(52413) 

8*(12) + 8*(21) 
1*(51342) -1*(51432) -1*(52341) + 1*(52431) 

8*(12) + 8*(21) 
1*(51234) -1*(51432) -1*(53214) + 1*(53412) 

8*(12) + 8*(21) 
1*(51243) -1*(51423) -1*(53241) + 1*(53421) 

8*(12) + 8*(21) 
1*(51324) -1*(51342) -1*(53124) + 1*(53142) 

8*(12) + 8*(21) 
1*(51234) -1*(51324) -1*(54231) + 1*(54321) 

8*(12) + 8*(21) 
1*(51243) -1*(51342) -1*(54213) + 1*(54312) 

8*(12) + 

8*(12) + 8*(21) 
1*(35241) -1*(35421) -1*(53241) + 1*(53421) 

8*(12) + 8*(21) 
1*(35412) -1*(35421) -1*(53412) + 1*(53421) 

8*(12) + 8*(21) 
1*(35124) -1*(35214) -1*(45123) + 1*(45213) 

8*(12) + 8*(21) 
1*(35142) -1*(35241) -1*(45132) + 1*(45231) 

8*(12) + 8*(21) 
1*(35412) -1*(35421) -1*(45312) + 1*(45321) 

8*(12) + 8*(21) 
1*(35124) -1*(35214) -1*(54123) + 1*(54213) 

8*(12) + 8*(21) 
1*(35142) -1*(35241) -1*(54132) + 1*(54231) 

8*(12) + 8*(21) 
1*(35412) -1*(35421) -1*(54312) + 1*(54321) 

8*(12) + 8*(21) 
1*(53124) -1*(53214) -1*(54123) + 1*(54213) 

8*(12) + 8*(21) 
1*(53142) -1*(53241) -1*(54132) + 1*(54231) 

8*(12) + 8*(21) 
1*(53412) -1*(53421) -1*(54312) + 1*(54321) 

8*(12) + 8*(21) 
1*(45123) -1*(45132) -1*(51423) + 1*(51432) 

8*(12) + 8*(21) 
1*(45213) -1*(45312) -1*(51243) + 1*(51342) 

8*(12) + 8*(21) 
1*(45231) -1*(45321) -1*(51234) + 1*(51324) 

8*(12) + 8*(21) 
1*(45123) -1*(45321) -1*(52143) + 1*(52341) 

8*(12) + 8*(21) 
1*(45132) -1*(45312) -1*(52134) + 1*(5

1*(12345) + 1*(52341) + 2*(2143) + 8*(132) + 8*(213) 
1*(12345) -1*(52341) + 2*(2143) + 6*(132) + 6*(213) + 6*(321) 

1*(12435) + 1*(52431) + 2*(2143) + 8*(132) + 8*(213) 
1*(12435) -1*(52431) + 2*(2143) + 6*(132) + 6*(213) + 6*(321) 

1*(12345) + 1*(52341) + 2*(3412) + 8*(123) 
1*(12345) -1*(52341) + 2*(3412) + 6*(123) + 6*(231) + 6*(312) 

1*(12435) + 1*(52431) + 2*(3412) + 8*(123) 
1*(12435) -1*(52431) + 2*(3412) + 6*(123) + 6*(231) + 6*(312) 

1*(13245) + 1*(53241) + 2*(2143) + 8*(132) + 8*(213) 
1*(13245) -1*(53241) + 2*(2143) + 6*(132) + 6*(213) + 6*(321) 

1*(13425) + 1*(53421) + 2*(2143) + 8*(132) + 8*(213) 
1*(13425) -1*(53421) + 2*(2143) + 6*(132) + 6*(213) + 6*(321) 

1*(13245) + 1*(53241) + 2*(3412) + 8*(123) 
1*(13245) -1*(53241) + 2*(3412) + 6*(123) + 6*(231) + 6*(312) 

1*(13425) + 1*(53421) + 2*(3412) + 8*(123) 
1*(13425) -1*(53421) + 2*(3412) + 6*(123) + 6*(231) + 6*(312) 

1*(14235) + 1*(54231) + 2*(2143) + 8*(132) + 8*(213) 
1*(14235) -1*(54231) + 2*(2143) + 6*(132) 

In [30]:
print(data_array)

[['[3, 3, 3, 3, 3, 3]', 1, 1], ['[3, 3, 3, 3, 3, 2]', 0, 10], ['[3, 3, 3, 3, 2, 2]', 0, 15], ['[e', 0, 0]]


for valid length tuple in valid length tuples:
    for number of rows up to max allowed rows to check:
        loop through all the partial permutations for each length
            flatten into vectors and solve as a system of linear equations
                if no solution is found:
                    continue
                if solution is found:
                    maybe print it at still possible.
                    this should probably be entered into a file for long term storage.

In [18]:
# Open a text file containing the length tuples we have searched already.
with open("v3checked_lengths_6.txt", "a+") as checked_lengths_file:
    checked_lengths_file.seek(0)
    checked_string = checked_lengths_file.read()
    #Loop through greatest n up to a max of 18
    for n in [18,16]:
        #Loop through possible lengths tuples for a given n
        for len_tuple in possibleLengths(n,6):
        #If we have already checked this tuple, skip this loop.
            if str(len_tuple) in checked_string:
                continue
            else:
                print(len_tuple)
                covers_found, search_data = searchRecur(len_tuple)
                if len(covers_found) > 0:
                    with open("v3found_covers_6.txt", "a") as found_covers_file:
                        found_covers_file.write(str(len_tuple)+"\n")
                        for cover in covers_found:
                            found_covers_file.write(str(cover)+"\n")
                checked_lengths_file.write(str(len_tuple)+"\n")
                if (search_data is not None) and (len(search_data) > 0):
                    with open("search_data_6.txt","a") as search_data_file:
                        search_data_file.write(str(len_tuple)+'\n')
                        for entry in search_data:
                            search_data_file.write(str(entry[0])+","+str(entry[1])+'\n')


[18, 18, 17, 15, 11, 7]
2026-05-29 23:15:29.745666
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[18, 18, 17, 15, 11, 6]
2026-05-30 00:13:20.486837
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[18, 18, 17, 15, 11, 5]
2026-05-30 01:04:22.419154
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[18, 18, 17, 15, 11, 4]
2026-05-30 01:47:40.412477
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[18, 18, 17, 15, 11, 3]
2026-05-30 02:23:46.283825
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[16, 16, 16, 16, 16, 16]
2026-05-30 02:52:17.325954
[16, 16, 16, 16, 16, 15]
2026-05-30 02:52:17.326601
[16, 16, 16

KeyboardInterrupt: 

In [4]:
for i,case in enumerate([*possibleLengths(18,6)]):
    print(str(i) + ": " + str(case))

0: [18, 18, 18, 18, 18, 18]
1: [18, 18, 18, 18, 18, 17]
2: [18, 18, 18, 18, 18, 16]
3: [18, 18, 18, 18, 18, 15]
4: [18, 18, 18, 18, 18, 14]
5: [18, 18, 18, 18, 17, 17]
6: [18, 18, 18, 18, 17, 16]
7: [18, 18, 18, 18, 17, 15]
8: [18, 18, 18, 18, 17, 14]
9: [18, 18, 18, 18, 17, 13]
10: [18, 18, 18, 18, 16, 16]
11: [18, 18, 18, 18, 16, 15]
12: [18, 18, 18, 18, 16, 14]
13: [18, 18, 18, 18, 16, 13]
14: [18, 18, 18, 18, 16, 12]
15: [18, 18, 18, 18, 15, 15]
16: [18, 18, 18, 18, 15, 14]
17: [18, 18, 18, 18, 15, 13]
18: [18, 18, 18, 18, 15, 12]
19: [18, 18, 18, 18, 15, 11]
20: [18, 18, 18, 17, 17, 17]
21: [18, 18, 18, 17, 17, 16]
22: [18, 18, 18, 17, 17, 15]
23: [18, 18, 18, 17, 17, 14]
24: [18, 18, 18, 17, 17, 13]
25: [18, 18, 18, 17, 17, 12]
26: [18, 18, 18, 17, 16, 16]
27: [18, 18, 18, 17, 16, 15]
28: [18, 18, 18, 17, 16, 14]
29: [18, 18, 18, 17, 16, 13]
30: [18, 18, 18, 17, 16, 12]
31: [18, 18, 18, 17, 16, 11]
32: [18, 18, 18, 17, 15, 15]
33: [18, 18, 18, 17, 15, 14]
34: [18, 18, 18, 17, 15,

In [6]:
for len_tuple in [[7, 7, 6, 6, 3, 3], [7, 7, 6, 6, 3, 2], [7, 7, 6, 6, 2, 2], [7, 7, 6, 5, 5, 5], [7, 7, 6, 5, 5, 4], [7, 7, 6, 5, 5, 3], [7, 7, 6, 5, 5, 2]]:
        print(len_tuple)
        covers_found, search_data = searchRecur(len_tuple)
        if len(covers_found) > 0:
            with open("data_collection_found_covers_6.txt", "a") as found_covers_file:
                found_covers_file.write(str(len_tuple)+"\n")
                for cover in covers_found:
                    found_covers_file.write(str(cover)+"\n")
        if (search_data is not None) and (len(search_data) > 0):
            with open("data_collection_search_data_6.txt","a") as search_data_file:
                search_data_file.write(str(len_tuple)+'\n')
                for entry in search_data:
                    search_data_file.write(str(entry[0])+","+str(entry[1])+'\n')

[7, 7, 6, 6, 3, 3]
Examining [0]
The 100th case: [[0], [0], [3], [4], [1], [1]]
The 200th case: [[0], [1], [2], [3], [0], [1]]
The 300th case: [[0], [2], [1], [2], [2], [2]]
The 400th case: [[0], [3], [0], [3], [1], [1]]
The 500th case: [[0], [3], [5], [5], [0], [1]]
The 600th case: [[0], [4], [3], [4], [0], [1]]
The 700th case: [[0], [5], [2], [3], [1], [1]]
The 800th case: [[0], [6], [1], [3], [0], [1]]
Examining [1]
The 900th case: [[1], [1], [0], [3], [2], [2]]
The 1000th case: [[1], [1], [5], [5], [1], [1]]
The 1100th case: [[1], [2], [3], [4], [0], [1]]
The 1200th case: [[1], [3], [2], [2], [2], [2]]
The 1300th case: [[1], [4], [1], [2], [1], [1]]
The 1400th case: [[1], [5], [0], [3], [0], [1]]
The 1500th case: [[1], [5], [5], [5], [1], [1]]
The 1600th case: [[1], [6], [3], [4], [2], [2]]
Examining [2]
The 1700th case: [[2], [2], [2], [3], [1], [1]]
The 1800th case: [[2], [3], [1], [3], [0], [1]]
The 1900th case: [[2], [4], [0], [3], [2], [2]]
The 2000th case: [[2], [4], [5], [5]

The 2900th case: [[0, 3], [1, 3], [5, 4], [1, 0], [1, 0], [2, 0]]
The 3000th case: [[0, 3], [1, 4], [5, 2], [1, 0], [1, 4], [2, 1]]
The 3100th case: [[0, 3], [1, 5], [5, 0], [1, 2], [1, 4], [2, 3]]
The 3200th case: [[0, 3], [1, 5], [5, 4], [1, 0], [1, 0], [2, 0]]
The 3300th case: [[0, 3], [1, 6], [5, 2], [1, 0], [1, 4], [2, 1]]
The 3400th case: [[0, 4], [1, 0], [5, 0], [1, 2], [1, 4], [2, 3]]
The 3500th case: [[0, 4], [1, 0], [5, 4], [1, 0], [1, 0], [2, 0]]
The 3600th case: [[0, 4], [1, 2], [5, 2], [1, 0], [1, 4], [2, 1]]
The 3700th case: [[0, 4], [1, 3], [5, 0], [1, 2], [1, 4], [2, 3]]
The 3800th case: [[0, 4], [1, 3], [5, 4], [1, 0], [1, 0], [2, 0]]
The 3900th case: [[0, 4], [1, 4], [5, 2], [1, 0], [1, 4], [2, 1]]
The 4000th case: [[0, 4], [1, 5], [5, 0], [1, 2], [1, 4], [2, 3]]
The 4100th case: [[0, 4], [1, 5], [5, 4], [1, 0], [1, 0], [2, 0]]
The 4200th case: [[0, 4], [1, 6], [5, 2], [1, 0], [1, 4], [2, 1]]
The 4300th case: [[0, 5], [1, 0], [5, 0], [1, 2], [1, 4], [2, 3]]
The 4400th

The 15400th case: [[0, 2], [6, 1], [1, 4], [2, 1], [2, 4], [3, 2]]
The 15500th case: [[0, 2, 1], [6, 2, 0], [1, 0, 5], [2, 1, 4], [2, 1, 4], [3, 2, 0]]
The 15600th case: [[0, 2, 1], [6, 2, 4], [1, 0, 2], [2, 1, 0], [2, 1, 3], [3, 2, 0]]
The 15700th case: [[0, 2, 3], [6, 2, 0], [1, 0, 2], [2, 1, 3], [2, 1, 3], [3, 2, 0]]
The 15800th case: [[0, 2, 3], [6, 2, 3], [1, 0, 2], [2, 1, 4], [2, 1, 4], [3, 2, 0]]
The 15900th case: [[0, 2, 3], [6, 2, 5], [1, 0, 3], [2, 1, 0], [2, 1, 3], [3, 2, 0]]
The 16000th case: [[0, 2, 4], [6, 2, 1], [1, 0, 3], [2, 1, 3], [2, 1, 3], [3, 2, 0]]
The 16100th case: [[0, 2, 4], [6, 2, 4], [1, 0, 3], [2, 1, 4], [2, 1, 4], [3, 2, 0]]
The 16200th case: [[0, 2, 5], [6, 2, 0], [1, 0, 4], [2, 1, 0], [2, 1, 3], [3, 2, 0]]
The 16300th case: [[0, 2, 5], [6, 2, 3], [1, 0, 4], [2, 1, 3], [2, 1, 3], [3, 2, 0]]
The 16400th case: [[0, 2, 5], [6, 2, 5], [1, 0, 4], [2, 1, 4], [2, 1, 4], [3, 2, 0]]
The 16500th case: [[0, 2, 6], [6, 2, 1], [1, 0, 5], [2, 1, 0], [2, 1, 3], [3, 2, 0]

The 26100th case: [[0, 1], [6, 2], [4, 0], [2, 0], [2, 3], [0, 1]]
The 26200th case: [[0, 1], [6, 2], [4, 3], [2, 1], [2, 3], [0, 2]]
The 26300th case: [[0, 1], [6, 3], [4, 1], [2, 3], [2, 4], [0, 3]]
The 26400th case: [[0, 1], [6, 4], [4, 0], [2, 0], [2, 3], [0, 1]]
The 26500th case: [[0, 1], [6, 4], [4, 3], [2, 1], [2, 3], [0, 2]]
The 26600th case: [[0, 1], [6, 5], [4, 1], [2, 3], [2, 4], [0, 3]]
The 26700th case: [[0, 2], [6, 0], [4, 0], [2, 0], [2, 3], [0, 1]]
The 26800th case: [[0, 2], [6, 0], [4, 3], [2, 1], [2, 3], [0, 2]]
The 26900th case: [[0, 2], [6, 1], [4, 1], [2, 3], [2, 4], [0, 3]]
The 27000th case: [[0, 2], [6, 2], [4, 0], [2, 0], [2, 3], [0, 1]]
The 27100th case: [[0, 2], [6, 2], [4, 3], [2, 1], [2, 3], [0, 2]]
The 27200th case: [[0, 2, 1], [6, 2, 1], [4, 5, 1], [2, 3, 1], [2, 3, 1], [0, 1, 2]]
The 27300th case: [[0, 2, 1], [6, 2, 4], [4, 5, 1], [2, 3, 4], [2, 3, 4], [0, 1, 2]]
The 27400th case: [[0, 2, 3], [6, 2, 0], [4, 5, 2], [2, 3, 0], [2, 3, 1], [0, 1, 2]]
The 2750

The 37500th case: [[1], [4], [5], [3], [4], [3]]
The 37600th case: [[1], [5], [1], [1], [4], [3]]
The 37700th case: [[1], [5], [3], [0], [4], [1]]
The 37800th case: [[1], [5], [4], [4], [4], [1]]
The 37900th case: [[1], [6], [0], [2], [2], [2]]
The 38000th case: [[1], [6], [2], [1], [1], [0]]
The 38100th case: [[1], [6], [4], [0], [0], [0]]
The 38200th case: [[1], [6], [5], [2], [3], [0]]
Examining [2]
The 38300th case: [[2], [2], [1], [1], [1], [0]]
The 38400th case: [[2], [2], [3], [0], [0], [0]]
The 38500th case: [[2], [2], [4], [2], [3], [0]]
The 38600th case: [[2], [3], [0], [1], [1], [0]]
The 38700th case: [[2], [3], [2], [0], [0], [0]]
The 38800th case: [[2], [3], [3], [2], [3], [0]]
The 38900th case: [[2], [3], [5], [1], [1], [1]]
The 39000th case: [[2], [4], [1], [0], [0], [1]]
The 39100th case: [[2], [4], [2], [2], [3], [1]]
The 39200th case: [[2], [4], [4], [1], [1], [1]]
The 39300th case: [[2], [5], [0], [0], [0], [1]]
The 39400th case: [[2], [5], [1], [2], [3], [1]]
The 39

The 51200th case: [[5, 3], [6, 1], [0, 1], [3, 1], [3, 2], [1, 2]]
The 51300th case: [[5, 3], [6, 1], [0, 4], [3, 2], [3, 4], [1, 3]]
The 51400th case: [[5, 3], [6, 2], [0, 3], [3, 0], [3, 2], [1, 0]]
The 51500th case: [[5, 3], [6, 3], [0, 1], [3, 1], [3, 2], [1, 2]]
The 51600th case: [[5, 3], [6, 3], [0, 4], [3, 2], [3, 4], [1, 3]]
The 51700th case: [[5, 3], [6, 4], [0, 3], [3, 0], [3, 2], [1, 0]]
The 51800th case: [[5, 3], [6, 5], [0, 1], [3, 1], [3, 2], [1, 2]]
The 51900th case: [[5, 3], [6, 5], [0, 4], [3, 2], [3, 4], [1, 3]]
The 52000th case: [[5, 4], [6, 0], [0, 3], [3, 0], [3, 2], [1, 0]]
The 52100th case: [[5, 4], [6, 1], [0, 1], [3, 1], [3, 2], [1, 2]]
The 52200th case: [[5, 4], [6, 1], [0, 4], [3, 2], [3, 4], [1, 3]]
The 52300th case: [[5, 4], [6, 2], [0, 3], [3, 0], [3, 2], [1, 0]]
The 52400th case: [[5, 4], [6, 3], [0, 1], [3, 1], [3, 2], [1, 2]]
The 52500th case: [[5, 4], [6, 3], [0, 4], [3, 2], [3, 4], [1, 3]]
The 52600th case: [[5, 4], [6, 4], [0, 3], [3, 0], [3, 2], [1,

The 11100th case: [[5, 0], [6, 4], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11200th case: [[5, 0], [6, 5], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11300th case: [[5, 1], [6, 0], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11400th case: [[5, 1], [6, 1], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11500th case: [[5, 1], [6, 2], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11600th case: [[5, 1], [6, 3], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11700th case: [[5, 1], [6, 4], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11800th case: [[5, 1], [6, 5], [0, 5], [2, 4], [2, 4], [1, 2]]
The 11900th case: [[5, 2], [6, 0], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12000th case: [[5, 2], [6, 1], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12100th case: [[5, 2], [6, 2], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12200th case: [[5, 2], [6, 3], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12300th case: [[5, 2], [6, 4], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12400th case: [[5, 2], [6, 5], [0, 5], [2, 4], [2, 4], [1, 2]]
The 12500th case: [[5, 3], [6, 0], [0, 5], [2, 4], [2, 4], [1,

The 9500th case: [[1], [6], [5], [3], [3], [0]]
Examining [2]
The 9600th case: [[2], [2], [3], [0], [2], [0]]
The 9700th case: [[2], [3], [0], [1], [3], [0]]
The 9800th case: [[2], [3], [3], [3], [3], [0]]
The 9900th case: [[2], [4], [1], [0], [2], [0]]
The 10000th case: [[2], [4], [4], [1], [3], [0]]
The 10100th case: [[2], [5], [1], [3], [4], [0]]
The 10200th case: [[2], [5], [5], [0], [3], [1]]
The 10300th case: [[2], [6], [2], [2], [2], [1]]
The 10400th case: [[2], [6], [5], [4], [4], [1]]
Examining [3]
The 10500th case: [[3], [3], [3], [0], [4], [1]]
The 10600th case: [[3], [4], [0], [2], [2], [1]]
The 10700th case: [[3], [4], [3], [4], [4], [1]]
The 10800th case: [[3], [5], [1], [0], [4], [1]]
The 10900th case: [[3], [5], [4], [2], [2], [1]]
The 11000th case: [[3], [6], [2], [0], [0], [0]]
The 11100th case: [[3], [6], [5], [1], [1], [1]]
Examining [4]
The 11200th case: [[4], [4], [2], [2], [3], [1]]
The 11300th case: [[4], [5], [0], [0], [0], [1]]
The 11400th case: [[4], [5], [3]

In [8]:
len_tuple = [3,3,3,3,3,3]
covers_found, search_data = searchRecur(len_tuple)
if len(covers_found) > 0:
    with open("v3found_covers_6.txt", "a") as found_covers_file:
        found_covers_file.write(str(len_tuple)+"\n")
        for cover in covers_found:
            found_covers_file.write(str(cover)+"\n")
if (search_data is not None) and (len(search_data) > 0):
    with open("search_data_6.txt","a") as search_data_file:
        search_data_file.write(str(len_tuple)+'\n')
        for entry in search_data:
            search_data_file.write(str(entry[0])+","+str(entry[1])+'\n')


2026-06-16 11:18:56.064387
Examining [0]
Found infFam with perms: [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]
Examining [1]


In [9]:
covers_found, search_data = searchRecur([5,5,5,5,5,2])

Examining [0]
The 10th case: [[0], [0], [0], [0], [4], [1]]
The 20th case: [[0], [0], [0], [2], [2], [1]]
The 30th case: [[0], [0], [0], [4], [4], [1]]
The 40th case: [[0], [0], [1], [2], [2], [1]]
The 50th case: [[0, 1], [0, 1], [1, 0], [2, 3], [3, 0], [1, 0]]
The 60th case: [[0, 1], [0, 1], [1, 2], [2, 0], [3, 2], [1, 0]]
The 70th case: [[0, 1], [0, 1], [1, 2], [2, 4], [3, 0], [1, 0]]
The 80th case: [[0, 1], [0, 1], [1, 3], [2, 1], [3, 2], [1, 0]]
The 90th case: [[0, 1], [0, 1], [1, 4], [2, 0], [3, 0], [1, 0]]
The 100th case: [[0, 1], [0, 1], [1, 4], [2, 3], [3, 2], [1, 0]]
The 110th case: [[0, 1], [0, 2], [1, 0], [2, 1], [3, 0], [1, 0]]
The 120th case: [[0, 1], [0, 2], [1, 0], [2, 4], [3, 2], [1, 0]]
The 130th case: [[0, 1], [0, 2], [1, 2], [2, 3], [3, 0], [1, 0]]
The 140th case: [[0, 1], [0, 2], [1, 3], [2, 0], [3, 2], [1, 0]]
The 150th case: [[0, 1], [0, 2], [1, 3], [2, 4], [3, 0], [1, 0]]
The 160th case: [[0, 1], [0, 2], [1, 4], [2, 1], [3, 2], [1, 0]]
The 170th case: [[0, 1], [0

The 1300th case: [[0, 4], [0, 4], [1, 3], [2, 3], [4, 2], [0, 1]]
The 1310th case: [[0, 4], [0, 4], [1, 4], [2, 1], [4, 0], [0, 1]]
The 1320th case: [[0, 4], [0, 4], [1, 4], [2, 4], [4, 2], [0, 1]]
The 1330th case: [[0, 1], [0, 1], [1, 0], [2, 3], [4, 0], [1, 0]]
The 1340th case: [[0, 1], [0, 1], [1, 2], [2, 0], [4, 2], [1, 0]]
The 1350th case: [[0, 1], [0, 1], [1, 2], [2, 4], [4, 0], [1, 0]]
The 1360th case: [[0, 1], [0, 1], [1, 3], [2, 1], [4, 2], [1, 0]]
The 1370th case: [[0, 1], [0, 1], [1, 4], [2, 0], [4, 0], [1, 0]]
The 1380th case: [[0, 1], [0, 1], [1, 4], [2, 3], [4, 2], [1, 0]]
The 1390th case: [[0, 1], [0, 2], [1, 0], [2, 1], [4, 0], [1, 0]]
The 1400th case: [[0, 1], [0, 2], [1, 0], [2, 4], [4, 2], [1, 0]]
The 1410th case: [[0, 1], [0, 2], [1, 2], [2, 3], [4, 0], [1, 0]]
The 1420th case: [[0, 1], [0, 2], [1, 3], [2, 0], [4, 2], [1, 0]]
The 1430th case: [[0, 1], [0, 2], [1, 3], [2, 4], [4, 0], [1, 0]]
The 1440th case: [[0, 1], [0, 2], [1, 4], [2, 1], [4, 2], [1, 0]]
The 1450th

The 2570th case: [[0, 4], [0, 4], [1, 2], [3, 4], [4, 2], [0, 1]]
The 2580th case: [[0, 4], [0, 4], [1, 3], [3, 2], [4, 0], [0, 1]]
The 2590th case: [[0, 4], [0, 4], [1, 4], [3, 0], [4, 2], [0, 1]]
The 2600th case: [[0, 4], [0, 4], [1, 4], [3, 4], [4, 0], [0, 1]]
The 2610th case: [[0, 1], [0, 1], [1, 0], [3, 1], [4, 2], [1, 0]]
The 2620th case: [[0, 1], [0, 1], [1, 2], [3, 0], [4, 0], [1, 0]]
The 2630th case: [[0, 1], [0, 1], [1, 2], [3, 2], [4, 2], [1, 0]]
The 2640th case: [[0, 1], [0, 1], [1, 3], [3, 1], [4, 0], [1, 0]]
The 2650th case: [[0, 1], [0, 1], [1, 3], [3, 4], [4, 2], [1, 0]]
The 2660th case: [[0, 1], [0, 1], [1, 4], [3, 2], [4, 0], [1, 0]]
The 2670th case: [[0, 1], [0, 2], [1, 0], [3, 0], [4, 2], [1, 0]]
The 2680th case: [[0, 1], [0, 2], [1, 0], [3, 4], [4, 0], [1, 0]]
The 2690th case: [[0, 1], [0, 2], [1, 2], [3, 1], [4, 2], [1, 0]]
The 2700th case: [[0, 1], [0, 2], [1, 3], [3, 0], [4, 0], [1, 0]]
The 2710th case: [[0, 1], [0, 2], [1, 3], [3, 2], [4, 2], [1, 0]]
The 2720th

The 3830th case: [[0, 4], [0, 4], [2, 0], [3, 0], [4, 0], [0, 1]]
The 3840th case: [[0, 4], [0, 4], [2, 0], [3, 2], [4, 2], [0, 1]]
The 3850th case: [[0, 4], [0, 4], [2, 1], [3, 1], [4, 0], [0, 1]]
The 3860th case: [[0, 4], [0, 4], [2, 1], [3, 4], [4, 2], [0, 1]]
The 3870th case: [[0, 4], [0, 4], [2, 3], [3, 2], [4, 0], [0, 1]]
The 3880th case: [[0, 4], [0, 4], [2, 4], [3, 0], [4, 2], [0, 1]]
The 3890th case: [[0, 4], [0, 4], [2, 4], [3, 4], [4, 0], [0, 1]]
The 3900th case: [[0, 1], [0, 1], [2, 0], [3, 1], [4, 2], [1, 0]]
The 3910th case: [[0, 1], [0, 1], [2, 1], [3, 0], [4, 0], [1, 0]]
The 3920th case: [[0, 1], [0, 1], [2, 1], [3, 2], [4, 2], [1, 0]]
The 3930th case: [[0, 1], [0, 1], [2, 3], [3, 1], [4, 0], [1, 0]]
The 3940th case: [[0, 1], [0, 1], [2, 3], [3, 4], [4, 2], [1, 0]]
The 3950th case: [[0, 1], [0, 1], [2, 4], [3, 2], [4, 0], [1, 0]]
The 3960th case: [[0, 1], [0, 2], [2, 0], [3, 0], [4, 2], [1, 0]]
The 3970th case: [[0, 1], [0, 2], [2, 0], [3, 4], [4, 0], [1, 0]]
The 3980th

The 5130th case: [[0, 4], [1, 2], [1, 3], [2, 4], [3, 4], [1, 0]]
The 5140th case: [[0, 4], [1, 2], [1, 4], [2, 3], [3, 1], [1, 0]]
The 5150th case: [[0, 4], [1, 3], [1, 3], [2, 0], [3, 4], [1, 0]]
The 5160th case: [[0, 4], [1, 3], [1, 3], [2, 4], [3, 1], [1, 0]]
The 5170th case: [[0, 4], [1, 3], [1, 4], [2, 1], [3, 4], [1, 0]]
The 5180th case: [[0, 4], [1, 4], [1, 4], [2, 0], [3, 1], [1, 0]]
The 5190th case: [[0, 4], [1, 4], [1, 4], [2, 3], [3, 4], [1, 0]]
The 5200th case: [[0, 1], [1, 0], [1, 0], [2, 1], [4, 1], [0, 1]]
The 5210th case: [[0, 1], [1, 0], [1, 0], [2, 4], [4, 3], [0, 1]]
The 5220th case: [[0, 1], [1, 0], [1, 2], [2, 3], [4, 1], [0, 1]]
The 5230th case: [[0, 1], [1, 0], [1, 3], [2, 0], [4, 3], [0, 1]]
The 5240th case: [[0, 1], [1, 0], [1, 3], [2, 4], [4, 1], [0, 1]]
The 5250th case: [[0, 1], [1, 0], [1, 4], [2, 1], [4, 3], [0, 1]]
The 5260th case: [[0, 1], [1, 2], [1, 2], [2, 0], [4, 1], [0, 1]]
The 5270th case: [[0, 1], [1, 2], [1, 2], [2, 3], [4, 3], [0, 1]]
The 5280th

...

The 17710th case: [[0, 2], [2, 1], [3, 0], [3, 0], [4, 3], [0, 1]]
The 17720th case: [[0, 2], [2, 1], [3, 0], [3, 4], [4, 1], [0, 1]]
The 17730th case: [[0, 2], [2, 1], [3, 1], [3, 2], [4, 3], [0, 1]]
The 17740th case: [[0, 2], [2, 1], [3, 2], [3, 4], [4, 1], [0, 1]]
The 17750th case: [[0, 2], [2, 3], [3, 0], [3, 0], [4, 3], [0, 1]]
The 17760th case: [[0, 2], [2, 3], [3, 0], [3, 4], [4, 1], [0, 1]]
The 17770th case: [[0, 2], [2, 3], [3, 1], [3, 2], [4, 3], [0, 1]]
The 17780th case: [[0, 2], [2, 3], [3, 2], [3, 4], [4, 1], [0, 1]]
The 17790th case: [[0, 2], [2, 4], [3, 0], [3, 0], [4, 3], [0, 1]]
The 17800th case: [[0, 2], [2, 4], [3, 0], [3, 4], [4, 1], [0, 1]]
The 17810th case: [[0, 2], [2, 4], [3, 1], [3, 2], [4, 3], [0, 1]]
The 17820th case: [[0, 2], [2, 4], [3, 2], [3, 4], [4, 1], [0, 1]]
The 17830th case: [[0, 3], [2, 0], [3, 0], [3, 0], [4, 3], [0, 1]]
The 17840th case: [[0, 3], [2, 0], [3, 0], [3, 4], [4, 1], [0, 1]]
The 17850th case: [[0, 3], [2, 0], [3, 1], [3, 2], [4, 3], [0,

The 18960th case: [[0, 2], [2, 0], [3, 1], [4, 0], [4, 3], [0, 1]]
The 18970th case: [[0, 2], [2, 0], [3, 2], [4, 0], [4, 3], [0, 1]]
The 18980th case: [[0, 2], [2, 0], [3, 4], [4, 0], [4, 3], [0, 1]]
The 18990th case: [[0, 2], [2, 1], [3, 0], [4, 0], [4, 3], [0, 1]]
The 19000th case: [[0, 2], [2, 1], [3, 1], [4, 0], [4, 3], [0, 1]]
The 19010th case: [[0, 2], [2, 1], [3, 2], [4, 0], [4, 3], [0, 1]]
The 19020th case: [[0, 2], [2, 1], [3, 4], [4, 0], [4, 3], [0, 1]]
The 19030th case: [[0, 2], [2, 3], [3, 0], [4, 0], [4, 3], [0, 1]]
The 19040th case: [[0, 2], [2, 3], [3, 1], [4, 0], [4, 3], [0, 1]]
The 19050th case: [[0, 2], [2, 3], [3, 2], [4, 0], [4, 3], [0, 1]]
The 19060th case: [[0, 2], [2, 3], [3, 4], [4, 0], [4, 3], [0, 1]]
The 19070th case: [[0, 2], [2, 4], [3, 0], [4, 0], [4, 3], [0, 1]]
The 19080th case: [[0, 2], [2, 4], [3, 1], [4, 0], [4, 3], [0, 1]]
The 19090th case: [[0, 2], [2, 4], [3, 2], [4, 0], [4, 3], [0, 1]]
The 19100th case: [[0, 2], [2, 4], [3, 4], [4, 0], [4, 3], [0,

The 20200th case: [[1, 0], [1, 2], [2, 1], [3, 4], [4, 1], [0, 1]]
The 20210th case: [[1, 0], [1, 2], [2, 3], [3, 1], [4, 3], [0, 1]]
The 20220th case: [[1, 0], [1, 2], [2, 4], [3, 0], [4, 1], [0, 1]]
The 20230th case: [[1, 0], [1, 2], [2, 4], [3, 2], [4, 3], [0, 1]]
The 20240th case: [[1, 0], [1, 3], [2, 0], [3, 1], [4, 1], [0, 1]]
The 20250th case: [[1, 0], [1, 3], [2, 0], [3, 4], [4, 3], [0, 1]]
The 20260th case: [[1, 0], [1, 3], [2, 1], [3, 2], [4, 1], [0, 1]]
The 20270th case: [[1, 0], [1, 3], [2, 3], [3, 0], [4, 3], [0, 1]]
The 20280th case: [[1, 0], [1, 3], [2, 3], [3, 4], [4, 1], [0, 1]]
The 20290th case: [[1, 0], [1, 3], [2, 4], [3, 1], [4, 3], [0, 1]]
The 20300th case: [[1, 0], [1, 4], [2, 0], [3, 0], [4, 1], [0, 1]]
The 20310th case: [[1, 0], [1, 4], [2, 0], [3, 2], [4, 3], [0, 1]]
The 20320th case: [[1, 0], [1, 4], [2, 1], [3, 1], [4, 1], [0, 1]]
The 20330th case: [[1, 0], [1, 4], [2, 1], [3, 4], [4, 3], [0, 1]]
The 20340th case: [[1, 0], [1, 4], [2, 3], [3, 2], [4, 1], [0,

The 21480th case: [[1, 0], [2, 1], [3, 2], [3, 2], [4, 1], [0, 1]]
The 21490th case: [[1, 0], [2, 1], [3, 4], [3, 4], [4, 3], [0, 1]]
The 21500th case: [[1, 0], [2, 3], [3, 0], [3, 2], [4, 1], [0, 1]]
The 21510th case: [[1, 0], [2, 3], [3, 1], [3, 1], [4, 3], [0, 1]]
The 21520th case: [[1, 0], [2, 3], [3, 2], [3, 2], [4, 1], [0, 1]]
The 21530th case: [[1, 0], [2, 3], [3, 4], [3, 4], [4, 3], [0, 1]]
The 21540th case: [[1, 0], [2, 4], [3, 0], [3, 2], [4, 1], [0, 1]]
The 21550th case: [[1, 0], [2, 4], [3, 1], [3, 1], [4, 3], [0, 1]]
The 21560th case: [[1, 0], [2, 4], [3, 2], [3, 2], [4, 1], [0, 1]]
The 21570th case: [[1, 0], [2, 4], [3, 4], [3, 4], [4, 3], [0, 1]]
The 21580th case: [[1, 2], [2, 0], [3, 0], [3, 2], [4, 1], [0, 1]]
The 21590th case: [[1, 2], [2, 0], [3, 1], [3, 1], [4, 3], [0, 1]]
The 21600th case: [[1, 2], [2, 0], [3, 2], [3, 2], [4, 1], [0, 1]]
The 21610th case: [[1, 2], [2, 0], [3, 4], [3, 4], [4, 3], [0, 1]]
The 21620th case: [[1, 2], [2, 1], [3, 0], [3, 2], [4, 1], [0,

The 22730th case: [[2], [3], [3], [4], [4], [1]]
Examining [3]
The 22740th case: [[3], [3], [3], [4], [4], [1]]
Examining [4]


In [ ]:
with open("sixTermFamiliesCleaned.txt","a") as cleaned_file:
    with open("v3found_covers_6.txt", "r") as dirty_file:
        for line in dirty_file:
            if "Infinite family:" in line:
                split_line = line[1:-1].split()
            else:
                cleaned_file.write(line)

In [19]:
searchRecur([4,4,3,2,2])

Examining [0]
Found infFam with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [0, 1, 2], [0, 1], [1, 0]]
Found infFam with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [0, 1, 2], [0, 1], [1, 0]]
Found infFam with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [2, 1, 0], [0, 1], [1, 0]]
Found infFam with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [2, 1, 0], [0, 1], [1, 0]]
Examining [1]
Found infFam with perms: [[1, 0, 3, 2], [2, 3, 0, 1], [0, 1, 2], [0, 1], [1, 0]]
Found infFam with perms: [[1, 3, 0, 2], [2, 0, 3, 1], [0, 1, 2], [0, 1], [1, 0]]
Found infFam with perms: [[1, 0, 3, 2], [2, 3, 0, 1], [2, 1, 0], [0, 1], [1, 0]]
Found infFam with perms: [[1, 3, 0, 2], [2, 0, 3, 1], [2, 1, 0], [0, 1], [1, 0]]
Examining [2]
Examining [3]


['infFam:1*(1234) + 1*(4321) -4/3*(123) + 1*(12) +t*(1*(1234) + 1*(4321) -4/3*(123) + 2/3*(12) -1/3*(21))',
 'infFam:1*(1324) + 1*(4231) -4/3*(123) + 1*(12) +t*(1*(1324) + 1*(4231) -4/3*(123) + 2/3*(12) -1/3*(21))',
 'infFam:-1/2*(1234) -1/2*(4321) + 2/3*(321) + 1/2*(12) +t*(1*(1234) + 1*(4321) -4/3*(321) -1/3*(12) + 2/3*(21))',
 'infFam:-1/2*(1324) -1/2*(4231) + 2/3*(321) + 1/2*(12) +t*(1*(1324) + 1*(4231) -4/3*(321) -1/3*(12) + 2/3*(21))',
 'infFam:1/3*(2143) + 1/3*(3412) + 4/9*(123) + 1/3*(21) +t*(1*(2143) + 1*(3412) + 4/3*(123) -1*(12))',
 'infFam:1/3*(2413) + 1/3*(3142) + 4/9*(123) + 1/3*(21) +t*(1*(2413) + 1*(3142) + 4/3*(123) -1*(12))',
 'infFam:1/3*(2143) + 1/3*(3412) + 4/9*(321) + 1/3*(12) +t*(1*(2143) + 1*(3412) + 4/3*(321) -1*(21))',
 'infFam:1/3*(2413) + 1/3*(3142) + 4/9*(321) + 1/3*(12) +t*(1*(2413) + 1*(3142) + 4/3*(321) -1*(21))']

# TEST BELOW

In [14]:
for r in range(2,6):
    for n in range(2,r+2):
        for len_tuple in possibleLengths(n,r):
            print(len_tuple)
            covers_found, search_data = searchRecur(len_tuple)
            if len(covers_found) > 0:
                for cover in covers_found:
                    print(cover)
            else:
                print("No covers found.")
            print()
            

[2, 2]
2026-05-09 16:02:51.947316
Covers of type [2, 2] are the latin squares of order 2.

[3, 3]
2026-05-09 16:02:51.948766
Covers of type [3, 3] cannot fill the first row with enough non-zero entries.

[3, 3, 3]
2026-05-09 16:02:51.949781
Covers of type [3, 3, 3] are the latin squares of order 3.

[3, 3, 2]
2026-05-09 16:02:51.950968
Examining [0]
Found cover with perms: [[0, 1, 2], [2, 1, 0], [0, 1]]
Found cover with perms: [[0, 1, 2], [2, 1, 0], [1, 0]]
Examining [1]
-2*(123) + 2*(321) + 3*(12) 

[4, 4, 4]
2026-05-09 16:02:52.018946
Covers of type [4, 4, 4] cannot fill the first row with enough non-zero entries.

[4, 4, 3]
2026-05-09 16:02:52.019338
Covers of type [4, 4, 3] are ruled out by repeated entries

[3, 3, 3, 3]
2026-05-09 16:02:52.020354
Covers of type [3, 3, 3, 3] are ruled out by length tuple alone.

[3, 3, 3, 2]
2026-05-09 16:02:52.020612
Examining [0]
Found cover with perms: [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 0]]
Examining [1]
Found cover with perms: [[1, 2, 0], [2

Found cover with perms: [[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 1, 2, 0], [0, 1, 2]]
Found cover with perms: [[0, 2, 1, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0], [0, 1, 2]]
Found cover with perms: [[0, 2, 1, 3], [1, 0, 3, 2], [2, 3, 1, 0], [3, 2, 0, 1], [0, 1, 2]]
Found cover with perms: [[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 1, 2, 0], [2, 1, 0]]
Found cover with perms: [[0, 1, 3, 2], [1, 0, 2, 3], [2, 3, 0, 1], [3, 1, 2, 0], [2, 1, 0]]
Found cover with perms: [[0, 2, 1, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0], [2, 1, 0]]
Found cover with perms: [[0, 1, 2, 3], [1, 0, 3, 2], [3, 1, 2, 0], [3, 2, 1, 0], [2, 1, 0]]
Found cover with perms: [[0, 1, 3, 2], [1, 0, 2, 3], [3, 1, 2, 0], [3, 2, 1, 0], [2, 1, 0]]
Found cover with perms: [[0, 2, 1, 3], [1, 0, 3, 2], [3, 1, 2, 0], [3, 2, 1, 0], [2, 1, 0]]
Found cover with perms: [[0, 1, 2, 3], [2, 3, 0, 1], [3, 1, 2, 0], [3, 2, 1, 0], [0, 1, 2]]
Found cover with perms: [[0, 2, 1, 3], [2, 3, 0, 1], [3, 1, 2, 0], [3, 2, 1, 0],

Found cover with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [1, 2, 0], [2, 0, 1], [2, 1, 0]]
Found cover with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [1, 2, 0], [2, 0, 1], [2, 1, 0]]
Examining [1]
3*(1234) + 3*(4321) -2*(123) + 2*(132) + 2*(213) 
3*(1324) + 3*(4231) -2*(123) + 2*(132) + 2*(213) 

[4, 4, 3, 3, 2]
2026-05-09 16:03:38.133402
Examining [0]
Found cover with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [0, 2, 1], [1, 0, 2], [0, 1]]
Found cover with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [0, 2, 1], [1, 0, 2], [0, 1]]
Found cover with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [0, 2, 1], [1, 0, 2], [1, 0]]
Found cover with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [0, 2, 1], [1, 0, 2], [1, 0]]
Found infFam with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [0, 1, 2], [2, 1, 0], [0, 1]]
Found infFam with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [0, 1, 2], [2, 1, 0], [0, 1]]
Found infFam with perms: [[0, 1, 2, 3], [3, 2, 1, 0], [0, 1, 2], [2, 1, 0], [1, 0]]
Found infFam with perms: [[0, 2, 1, 3], [3, 1, 2, 0], [0, 1, 2], [2, 1, 

Found cover with perms: [[1, 0, 2, 4, 3], [3, 4, 2, 0, 1], [0, 3, 2, 1], [2, 1, 0, 3], [1, 0]]
Found cover with perms: [[1, 0, 2, 4, 3], [3, 4, 2, 0, 1], [1, 0, 3, 2], [3, 2, 1, 0], [0, 1]]
Found cover with perms: [[1, 0, 2, 4, 3], [3, 4, 2, 0, 1], [1, 2, 3, 0], [3, 0, 1, 2], [0, 1]]
Examining [2]
24*(12345) -24*(54321) -15*(1234) + 15*(4321) + 5*(12) 
-24*(12345) + 24*(52341) -5*(2143) + 5*(3412) + 5*(12) 
-24*(12435) + 24*(52431) -5*(2143) + 5*(3412) + 5*(12) 
-24*(13425) + 24*(53421) -5*(2143) + 5*(3412) + 5*(12) 
-24*(14325) + 24*(54321) -5*(2143) + 5*(3412) + 5*(12) 
24*(21354) -24*(45312) + 15*(1234) + 15*(3412) + 5*(21) 
24*(21354) -24*(45312) + 15*(1432) + 15*(3214) + 5*(21) 

[5, 5, 4, 3, 3]
2026-05-09 16:07:07.303048
Examining [0]
Found cover with perms: [[0, 1, 2, 3, 4], [4, 3, 2, 1, 0], [0, 1, 2, 3], [0, 1, 2], [2, 1, 0]]
Found cover with perms: [[0, 1, 2, 3, 4], [4, 1, 2, 3, 0], [1, 0, 3, 2], [0, 2, 1], [1, 0, 2]]
Found cover with perms: [[0, 1, 3, 2, 4], [4, 1, 3, 2, 0], 

KeyboardInterrupt: 

In [6]:
for n in range(7,11):
    for len_tuple in possibleLengths(n,5):
        print(len_tuple)
        covers_found, search_data = searchRecur(len_tuple)
        if len(covers_found) > 0:
            for cover in covers_found:
                print(cover)
        else:
            print("No covers found.")
        print()


[7, 7, 7, 7, 7]
Covers of type [7, 7, 7, 7, 7] cannot fill the first row with enough non-zero entries.

[7, 7, 7, 7, 6]
Covers of type [7, 7, 7, 7, 6] cannot fill the first row with enough non-zero entries.

[7, 7, 7, 7, 5]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
[7, 7, 7, 7, 5]: No covers found on full search.

[7, 7, 7, 7, 4]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
[7, 7, 7, 7, 4]: No covers found on full search.

[7, 7, 7, 6, 6]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
[7, 7, 7, 6, 6]: No covers found on full search.

[7, 7, 7, 6, 5]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
[7, 7, 7, 6, 5]: No covers found on full search.

[7, 7, 7, 6, 4]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
[7, 7, 7, 6, 4]: No co

Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[9, 9, 9, 8, 6]: No covers found on full search.

[9, 9, 9, 8, 5]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[9, 9, 9, 8, 5]: No covers found on full search.

[9, 9, 9, 7, 7]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[9, 9, 9, 7, 7]: No covers found on full search.

[9, 9, 9, 7, 6]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[9, 9, 9, 7, 6]: No covers found on full search.

[9, 9, 9, 7, 5]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
[9, 9, 9, 7, 5]: No covers found on full search.

[9, 9, 9, 7, 4]
Examining [0]
Examining [1]
Examining [2]
Examining [3

Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
Examining [9]
[10, 10, 9, 7, 4]: No covers found on full search.

[10, 10, 9, 7, 3]
Examining [0]
Examining [1]
Examining [2]
Examining [3]
Examining [4]
Examining [5]
Examining [6]
Examining [7]
Examining [8]
Examining [9]
[10, 10, 9, 7, 3]: No covers found on full search.



In [13]:
infFam = [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [0, 1]]
n = len(infFam[0])
rows = [genPermMatrixByRow(p,n,n,len(p)).list() for p in infFam]
M = matrix(QQ,rows).transpose()
M_aug = M.augment(vector(QQ,[1]*n**2))

In [43]:
print(M_aug.echelon_form().pivots())
print(M_aug.echelon_form())
print(M.right_kernel())
print(M.right_kernel().basis())
cover = combination_of_permutations(list(M.right_kernel().basis()[0]),infFam)
print(cover)

particularCover = combination_of_permutations(list(M.solve_right(vector(QQ, [1]*n**2))),infFam)

print(cover.isConstant(3))
print(particularCover.isConstant(3))

(0, 1, 2, 3, 4)
[  1   0   0   0   0 2/3   1]
[  0   1   0   0   0 2/3   0]
[  0   0   1   0   0 2/3   0]
[  0   0   0   1   0   0   1]
[  0   0   0   0   1   0   1]
[  0   0   0   0   0   0   0]
[  0   0   0   0   0   0   0]
[  0   0   0   0   0   0   0]
[  0   0   0   0   0   0   0]
Vector space of degree 6 and dimension 1 over Rational Field
Basis matrix:
[   1    1    1    0    0 -3/2]
[
(1, 1, 1, 0, 0, -3/2)
]
1*(123) + 1*(132) + 1*(213) 0*(231) 0*(312) -3/2*(12) 
True
True


In [56]:
infFam = [[1, 4, 2, 0, 3], [3, 0, 2, 4, 1], [1, 3, 0, 2], [2, 1, 0], [0, 1], [1, 0]]
n = len(infFam[0])
rows = [genPermMatrixByRow(p,n,n,len(p)).list() for p in infFam]
M = matrix(QQ,rows).transpose()
print(infiniteFamilyBasis(infFam,M))

-1/4*(25314) + 1/4*(41352) + 5/24*(2413) + 5/36*(321) + 5/48*(12) +t*(1*(25314) -1*(41352) -5/6*(2413) -5/9*(321) + 5/12*(21))


In [53]:
len_tuple = [5,5,4,3]
covers_found = searchRecur(len_tuple)
if len(covers_found) > 0:
    print(len_tuple)
    for cover in covers_found:
        print(cover)
else:
    print(len_tuple)
    print("No covers found.")
print()

[(0, 1), (1, 0), (3, 0), (1, 0)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (12/5, 3/5, 0, 1/5, 4/5), (18/5, 18/5, 3, 9/5, 0)]
[(0, 1), (1, 0), (3, 0), (1, 2)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (12/5, 3/5, 0, 1/5, 4/5), (0, 9/5, 3, 18/5, 18/5)]
[(0, 1), (1, 0), (3, 1), (1, 0)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (0, 9/5, 6/5, 1/5, 4/5), (18/5, 18/5, 3, 9/5, 0)]
[(0, 1), (1, 0), (3, 1), (1, 2)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (0, 9/5, 6/5, 1/5, 4/5), (0, 9/5, 3, 18/5, 18/5)]
[(0, 1), (1, 0), (3, 2), (1, 0)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (0, 0, 6/5, 2, 4/5), (18/5, 18/5, 3, 9/5, 0)]
[(0, 1), (1, 0), (3, 2), (1, 2)]
[(0, 1, 0, 0, 0), (1, 0, 0, 0, 0), (0, 0, 6/5, 2, 4/5), (0, 9/5, 3, 18/5, 18/5)]
[(0, 1), (1, 2), (3, 0), (1, 0)]
[(0, 1, 0, 0, 0), (0, 0, 1, 0, 0), (12/5, 3/5, 0, 1/5, 4/5), (18/5, 18/5, 3, 9/5, 0)]
[(0, 1), (1, 2), (3, 0), (1, 2)]
[(0, 1, 0, 0, 0), (0, 0, 1, 0, 0), (12/5, 3/5, 0, 1/5, 4/5), (0, 9/5, 3, 18/5, 18/5)]
[(0, 1), (1, 2), (3, 1), (1, 0)]
[(0, 1, 0, 0, 0), (0, 0, 1,

[(0, 1, 2, 3), (4, 1, 3, 2), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 2, 3), (4, 1, 3, 2), (1, 0, 3, 2), (2, 1, 0)]
[(0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 2, 4), (4, 1, 0, 2), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 1, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 2, 4), (4, 1, 0, 2), (1, 0, 3, 2), (2, 1, 0)]
[(0, 0, 1, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 2, 4), (4, 1, 0, 3), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 1, 0, 0, 0, 0, 0, 0, 1), (1, 0, 0, 0, 0, 0, 0, 0, 1, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 

[(0, 1, 4, 3), (4, 1, 0, 3), (1, 0, 3, 2), (2, 1, 0)]
[(0, 0, 0, 0, 1, 0, 0, 0, 1, 0), (1, 0, 0, 0, 0, 0, 0, 0, 1, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 4, 3), (4, 1, 2, 0), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 4, 3), (4, 1, 2, 0), (1, 0, 3, 2), (2, 1, 0)]
[(0, 0, 0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 4, 3), (4, 1, 2, 3), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 1, 4, 3), (4, 1, 2, 3), (1, 0, 3, 2), (2, 1, 0)]
[(0, 0, 0, 0, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 

[(0, 2, 1, 4), (4, 2, 3, 0), (1, 0, 2, 3), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 2, 1, 4), (4, 2, 3, 0), (1, 0, 3, 2), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 2, 1, 4), (4, 2, 3, 1), (1, 0, 2, 3), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 2, 1, 4), (4, 2, 3, 1), (1, 0, 3, 2), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 2, 3, 1), (4, 2, 0, 1), (1, 0, 2, 3), (2, 1, 0)]
[(0, 0, 0, 1, 0, 0, 1, 0, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 

[(0, 3, 1, 2), (4, 3, 0, 1), (1, 0, 2, 3), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 3, 1, 2), (4, 3, 0, 1), (1, 0, 3, 2), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 3, 1, 2), (4, 3, 0, 2), (1, 0, 2, 3), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 2, 4/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 3, 1, 2), (4, 3, 0, 2), (1, 0, 3, 2), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 0, 0, 2/5, 6/5, 12/5), (6/5, 3, 18/5, 3, 6/5, 18/5, 18/5, 3, 9/5, 0)]
[(0, 3, 1, 2), (4, 3, 1, 0), (1, 0, 2, 3), (2, 1, 0)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 4/5, 6/5, 0, 0, 0, 6/5, 

...

[(0, 1, 4, 2), (4, 1, 2, 3), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 1, 0, 0, 0, 0, 0, 1, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 1, 4, 2), (4, 1, 3, 0), (2, 3, 0, 1), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 1, 4, 2), (4, 1, 3, 0), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 1, 4, 2), (4, 1, 3, 2), (2, 3, 0, 1), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 1, 4, 2), (4, 1, 3, 2), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 0, 1, 0, 0), (0, 0, 0, 1, 0, 0, 0, 1, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5

[(0, 2, 1, 4), (4, 2, 1, 0), (2, 3, 1, 0), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 2, 1, 4), (4, 2, 1, 3), (2, 3, 0, 1), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0, 1, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 2, 1, 4), (4, 2, 1, 3), (2, 3, 1, 0), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 1, 0, 0, 0, 0, 0, 0, 1, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 2, 1, 4), (4, 2, 3, 0), (2, 3, 0, 1), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 2, 1, 4), (4, 2, 3, 0), (2, 3, 1, 0), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 0, 0, 1), (0, 0, 0, 1, 0, 1, 0, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5

[(0, 3), (4, 3), (2, 0), (0, 1)]
[(0, 0, 0, 1, 0), (0, 0, 0, 1, 0), (12/5, 3/5, 2/5, 3/5, 0), (18/5, 18/5, 3, 9/5, 0)]
[(0, 3), (4, 3), (2, 0), (0, 2)]
[(0, 0, 0, 1, 0), (0, 0, 0, 1, 0), (12/5, 3/5, 2/5, 3/5, 0), (18/5, 9/5, 6/5, 9/5, 18/5)]
[(0, 3), (4, 3), (2, 1), (0, 1)]
[(0, 0, 0, 1, 0), (0, 0, 0, 1, 0), (0, 9/5, 8/5, 3/5, 0), (18/5, 18/5, 3, 9/5, 0)]
[(0, 3), (4, 3), (2, 1), (0, 2)]
[(0, 0, 0, 1, 0), (0, 0, 0, 1, 0), (0, 9/5, 8/5, 3/5, 0), (18/5, 9/5, 6/5, 9/5, 18/5)]
[(0, 3), (4, 3), (2, 3), (0, 1)]
[(0, 0, 0, 1, 0), (0, 0, 0, 1, 0), (0, 0, 2/5, 6/5, 12/5), (18/5, 18/5, 3, 9/5, 0)]
[(0, 3, 1, 2), (4, 3, 0, 1), (2, 3, 0, 1), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 3, 1, 2), (4, 3, 0, 1), (2, 3, 1, 0), (0, 1, 2)]
[(0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (1, 0, 0, 0, 0, 0, 1, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5

[(0, 3, 4, 1), (4, 3, 1, 0), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 1, 0, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 3, 4, 1), (4, 3, 1, 2), (2, 3, 0, 1), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 3, 4, 1), (4, 3, 1, 2), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 1, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 3, 4, 1), (4, 3, 2, 0), (2, 3, 0, 1), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 1, 0, 0, 0, 0), (8/5, 2/5, 0, 2/5, 8/5, 12/5, 6/5, 2/5, 0, 0), (6/5, 3, 18/5, 3, 6/5, 0, 9/5, 3, 18/5, 18/5)]
[(0, 3, 4, 1), (4, 3, 2, 0), (2, 3, 1, 0), (0, 1, 2)]
[(0, 0, 0, 0, 1, 0, 1, 0, 0, 0), (0, 0, 1, 0, 0, 1, 0, 0, 0, 0), (0, 6/5, 4/5, 2/5, 8/5, 4/5, 2, 6/5

[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 2, 6/5, 0, 0), (18/5, 18/5, 3, 9/5, 0)]
[(3, 2), (4, 0), (0, 1), (1, 2)]
[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 2, 6/5, 0, 0), (0, 9/5, 3, 18/5, 18/5)]
[(3, 2), (4, 0), (0, 2), (1, 0)]
[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 1/5, 6/5, 9/5, 0), (18/5, 18/5, 3, 9/5, 0)]
[(3, 2), (4, 0), (0, 2), (1, 2)]
[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 1/5, 6/5, 9/5, 0), (0, 9/5, 3, 18/5, 18/5)]
[(3, 2), (4, 0), (0, 3), (1, 0)]
[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 1/5, 0, 3/5, 12/5), (18/5, 18/5, 3, 9/5, 0)]
[(3, 2), (4, 0), (0, 3), (1, 2)]
[(0, 0, 1, 0, 0), (1, 0, 0, 0, 0), (4/5, 1/5, 0, 3/5, 12/5), (0, 9/5, 3, 18/5, 18/5)]
[(3, 2), (4, 1), (0, 1), (1, 0)]
[(0, 0, 1, 0, 0), (0, 1, 0, 0, 0), (4/5, 2, 6/5, 0, 0), (18/5, 18/5, 3, 9/5, 0)]
[(3, 2), (4, 1), (0, 1), (1, 2)]
[(0, 0, 1, 0, 0), (0, 1, 0, 0, 0), (4/5, 2, 6/5, 0, 0), (0, 9/5, 3, 18/5, 18/5)]
[(3, 2), (4, 1), (0, 2), (1, 0)]
[(0, 0, 1, 0, 0), (0, 1, 0, 0, 0), (4/5, 1/5, 6/5, 9/5, 0), (18/5, 18/

In [106]:
for n in range(3,7):
    for k1 in range(2,n+1):
        print(f"({n},{k1}) : {findRepeatedEntries(n,k1)}")

for n in range(3,7):
    for k1 in range(2,n+1):
        for k2 in range(2,k1+1):
            print(f"({n},{k1},{k2}) : {findRepeatedEntriesPair(n,k1,k2)}")

(3,2) : (5, [[1, 2], [2, 1]])
(3,3) : (3, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
(4,2) : (4, [[1, 2], [2, 1]])
(4,3) : (4, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
(4,4) : (4, [[1, 2, 3, 4], [1, 2, 4, 3], [1, 3, 2, 4], [1, 3, 4, 2], [1, 4, 2, 3], [1, 4, 3, 2], [2, 1, 3, 4], [2, 1, 4, 3], [2, 3, 1, 4], [2, 3, 4, 1], [2, 4, 1, 3], [2, 4, 3, 1], [3, 1, 2, 4], [3, 1, 4, 2], [3, 2, 1, 4], [3, 2, 4, 1], [3, 4, 1, 2], [3, 4, 2, 1], [4, 1, 2, 3], [4, 1, 3, 2], [4, 2, 1, 3], [4, 2, 3, 1], [4, 3, 1, 2], [4, 3, 2, 1]])
(5,2) : (9, [[1, 2], [2, 1]])
(5,3) : (7, [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
(5,4) : (4, [[1, 2, 3, 4], [1, 3, 2, 4], [1, 4, 3, 2], [2, 1, 4, 3], [2, 3, 4, 1], [2, 4, 1, 3], [3, 1, 4, 2], [3, 2, 1, 4], [3, 4, 1, 2], [4, 1, 2, 3], [4, 2, 3, 1], [4, 3, 2, 1]])
(5,5) : (5, [[1, 2, 3, 4, 5], [1, 2, 3, 5, 4], [1, 2, 4, 3, 5], [1, 2, 4, 5, 3], [1, 2, 5, 3, 4], [1, 2, 5, 4, 3], [1, 3, 2, 4, 5], [1, 3,

(4,3,2) : (8, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])])
(4,3,3) : (8, [([1, 2, 3], [3, 2, 1]), ([3, 2, 1], [1, 2, 3])])
(4,4,2) : (8, [([2, 1, 4, 3], [1, 2]), ([2, 1, 4, 3], [2, 1]), ([3, 4, 1, 2], [1, 2]), ([3, 4, 1, 2], [2, 1])])
(4,4,3) : (8, [([2, 1, 4, 3], [3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3])])
(4,4,4) : (8, [([1, 2, 3, 4], [2, 1, 4, 3]), ([1, 2, 3, 4], [2, 3, 4, 1]), ([1, 2, 3, 4], [2, 4, 1, 3]), ([1, 2, 3, 4], [3, 1, 4, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 2, 1]), ([1, 2, 3, 4], [4, 1, 2, 3]), ([1, 2, 3, 4], [4, 3, 1, 2]), ([1, 2, 3, 4], [4, 3, 2, 1]), ([1, 2, 4, 3], [2, 1, 3, 4]), ([1, 2, 4, 3], [2, 3, 1, 4]), ([1, 2, 4, 3], [2, 4, 3, 1]), ([1, 2, 4, 3], [3, 1, 2, 4]), ([1, 2, 4, 3], [3, 4, 1, 2]), ([1, 2, 4, 3], [3, 4, 2, 1]), ([1, 2, 4, 3], [4, 1, 3, 2]), ([1, 2, 4, 3], [4, 3, 1, 2]), ([1, 2, 4, 3], [4, 3, 2, 1]), ([1, 3, 2, 4], [2, 1, 4, 3]), ([1, 3, 2, 4], [2, 4, 1, 3]), ([1, 3, 2, 4], [2, 4, 3, 1]), ([1, 3, 2

(5,5,2) : (12, [([2, 1, 3, 5, 4], [1, 2]), ([2, 1, 3, 5, 4], [1, 2]), ([2, 1, 3, 5, 4], [2, 1]), ([4, 5, 3, 1, 2], [1, 2]), ([4, 5, 3, 1, 2], [2, 1]), ([4, 5, 3, 1, 2], [2, 1])])
(5,5,3) : (10, [([1, 2, 3, 4, 5], [2, 3, 1]), ([1, 2, 3, 4, 5], [3, 1, 2]), ([1, 3, 2, 5, 4], [3, 2, 1]), ([1, 4, 3, 2, 5], [1, 3, 2]), ([1, 4, 3, 2, 5], [2, 1, 3]), ([1, 5, 3, 4, 2], [2, 1, 3]), ([1, 5, 4, 3, 2], [2, 1, 3]), ([2, 1, 3, 5, 4], [3, 2, 1]), ([2, 1, 4, 3, 5], [3, 2, 1]), ([2, 1, 4, 5, 3], [3, 2, 1]), ([2, 1, 5, 3, 4], [3, 2, 1]), ([2, 3, 1, 5, 4], [3, 2, 1]), ([2, 3, 4, 5, 1], [3, 1, 2]), ([2, 4, 3, 5, 1], [3, 1, 2]), ([3, 1, 2, 5, 4], [3, 2, 1]), ([3, 5, 4, 1, 2], [1, 2, 3]), ([4, 2, 3, 1, 5], [1, 3, 2]), ([4, 3, 2, 1, 5], [1, 3, 2]), ([4, 3, 5, 1, 2], [1, 2, 3]), ([4, 5, 1, 3, 2], [1, 2, 3]), ([4, 5, 2, 1, 3], [1, 2, 3]), ([4, 5, 2, 3, 1], [1, 2, 3]), ([4, 5, 3, 1, 2], [1, 2, 3]), ([5, 1, 2, 3, 4], [2, 3, 1]), ([5, 1, 3, 2, 4], [2, 3, 1]), ([5, 2, 3, 4, 1], [2, 3, 1]), ([5, 2, 3, 4, 1], [3, 1, 

(6,3,2) : (10, [([1, 2, 3], [1, 2]), ([1, 2, 3], [2, 1]), ([3, 2, 1], [1, 2]), ([3, 2, 1], [2, 1])])
(6,3,3) : (10, [([1, 2, 3], [3, 2, 1]), ([3, 2, 1], [1, 2, 3])])
(6,4,2) : (12, [([2, 1, 4, 3], [1, 2]), ([2, 1, 4, 3], [2, 1]), ([3, 4, 1, 2], [1, 2]), ([3, 4, 1, 2], [2, 1])])
(6,4,3) : (18, [([2, 1, 4, 3], [3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3])])
(6,4,4) : (12, [([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [3, 4, 1, 2]), ([1, 2, 3, 4], [4, 2, 3, 1]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [3, 4, 1, 2]), ([1, 3, 2, 4], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 2, 3, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([2, 1, 4, 3], [4, 3, 2, 1]), ([3, 4, 1, 2], [1, 2, 3, 4]), ([3,

KeyboardInterrupt: 

In [7]:
nIs11 = [*possibleLengths(11,6)]
nIs10 = [*possibleLengths(10,6)]
nIs9 = [*possibleLengths(9,6)]

print(nIs10.index([10, 10, 9, 7, 3, 2]))
print(len(nIs10))

print(nIs11.index([11, 11, 10, 9, 9, 3]))
print(len(nIs11))

print(nIs9.index([9, 9, 8, 7, 6, 6]))
print(len(nIs9))


nIs13 = [*possibleLengths(13,6)]
print(nIs13.index([13, 13, 13, 12, 10, 6]))
print(len(nIs13))

138
139
107
156
90
119
36
179


In [68]:
rho_star.sort(True)
for m in range(1,5):
    matrix_list = []
    count = 0
    for perm in rho_star.get_terms():

        slice_perm  = list(perm)[:m]
        matrix_list.append(vector(QQ, genPermMatrixByRow(slice_perm,4,m,len(perm)).list()))
        count += 1
        print(slice_perm)
    Amat = matrix(QQ,matrix_list)
    print(Amat)

    ones = vector(QQ,[1]*(4*m))
    try:
        solnones = Amat.solve_left(ones)
        print(solnones)   
    except ValueError:
        print("zinks")

[2]
[2]
[1]
[1]
[2]
[0]
[  0   0   1   0]
[  0   0   1   0]
[  0   1   0   0]
[  0   1   0   0]
[  0   0 3/4 9/4]
[9/4 3/4   0   0]
(2/3, 0, 2/3, 0, 4/9, 4/9)
[2, 3]
[2, 0]
[1, 3]
[1, 0]
[2, 1]
[0, 1]
[  0   0   1   0   0   0   0   1]
[  0   0   1   0   1   0   0   0]
[  0   1   0   0   0   0   0   1]
[  0   1   0   0   1   0   0   0]
[  0   0 3/4 9/4   0   1 5/4 3/4]
[9/4 3/4   0   0 3/4 5/4   1   0]
(0, 2/3, 2/3, 0, 4/9, 4/9)
[2, 3, 0]
[2, 0, 3]
[1, 3, 0]
[1, 0, 3]
[2, 1, 0]
[0, 1, 2]
[  0   0   1   0   0   0   0   1   1   0   0   0]
[  0   0   1   0   1   0   0   0   0   0   0   1]
[  0   1   0   0   0   0   0   1   1   0   0   0]
[  0   1   0   0   1   0   0   0   0   0   0   1]
[  0   0 3/4 9/4   0   1 5/4 3/4 3/4 5/4   1   0]
[9/4 3/4   0   0 3/4 5/4   1   0   0   1 5/4 3/4]
(0, 2/3, 2/3, 0, 4/9, 4/9)
[2, 3, 0, 1]
[2, 0, 3, 1]
[1, 3, 0, 2]
[1, 0, 3, 2]
[2, 1, 0]
[0, 1, 2]
[  0   0   1   0   0   0   0   1   1   0   0   0   0   1   0   0]
[  0   0   1   0   1   0   0   0   0   0   

In [57]:
print(genPermPolynomialByRow(j_perm([1,2]),3,3,2))
R.<x,y> = PolynomialRing(QQ)
k=2
p=j_perm([1,2])
for thing in [binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(2)]:
    print(thing)


2/3*x*y - 2/3*x - 2/3*y + 4/3
x*y - 2*x - 2*y + 4
x*y


In [69]:
R.<x,y> = PolynomialRing(QQ)
f = 0
for pair in rho_star.nested_list():
    p = pair[1]
    k = len(list(p))
    n = rho_star.order()
    f += pair[0]*factorial(n-k)/binomial(n,k)*sum(binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(k))
    for i in range(4):
        for j in range(4):
            print(f(i,j),end=" ")
        print()
    print()
    
print(f)

0 0 1 0 
0 0 0 1 
1 0 0 0 
0 1 0 0 

0 0 3/2 0 
1/2 0 0 1 
1 0 0 1/2 
0 3/2 0 0 

0 1/2 3/2 0 
1/2 0 0 3/2 
3/2 0 0 1/2 
0 3/2 1/2 0 

0 3/2 3/2 0 
3/2 0 0 3/2 
3/2 0 0 3/2 
0 3/2 3/2 0 

0 3/2 9/4 9/4 
3/2 1 5/4 9/4 
9/4 5/4 1 3/2 
9/4 9/4 3/2 0 

9/4 9/4 9/4 9/4 
9/4 9/4 9/4 9/4 
9/4 9/4 9/4 9/4 
9/4 9/4 9/4 9/4 

9/4


In [105]:
for mat in rho_star.genFuzzyMatrixList(4):
    print(mat)
    print()

[9/4 3/4   0   0]
[3/4 5/4   1   0]
[  0   1 5/4 3/4]
[  0   0 3/4 9/4]

[  0   0 3/4 9/4]
[  0   1 5/4 3/4]
[3/4 5/4   1   0]
[9/4 3/4   0   0]

[0 1 0 0]
[1 0 0 0]
[0 0 0 1]
[0 0 1 0]

[0 0 1 0]
[0 0 0 1]
[1 0 0 0]
[0 1 0 0]

[  0 1/2   0   0]
[  0   0   0 1/2]
[1/2   0   0   0]
[  0   0 1/2   0]

[  0   0 1/2   0]
[1/2   0   0   0]
[  0   0   0 1/2]
[  0 1/2   0   0]



In [99]:
print(rho_star.genFuzzyMatrix(4))
print(rho_star)

[9/4 9/4 9/4 9/4]
[9/4 9/4 9/4 9/4]
[9/4 9/4 9/4 9/4]
[9/4 9/4 9/4 9/4]
1*(123) + 1*(321) + 1*(2143) + 1*(3412) + 1/2*(2413) + 1/2*(3142) 


In [11]:
for perm in Permutations(5):
    print(genPermPolynomialByRow(j_perm(perm),5,5,5))
    print()

35/288*x^4*y^4 - 35/36*x^4*y^3 - 35/36*x^3*y^4 + 685/288*x^4*y^2 + 565/72*x^3*y^3 + 685/288*x^2*y^4 - 125/72*x^4*y - 175/9*x^3*y^2 - 175/9*x^2*y^3 - 125/72*x*y^4 + 1/24*x^4 + 1043/72*x^3*y + 14147/288*x^2*y^2 + 1043/72*x*y^3 + 1/24*y^4 - 5/12*x^3 - 2725/72*x^2*y - 2725/72*x*y^2 - 5/12*y^3 + 35/24*x^2 + 2245/72*x*y + 35/24*y^2 - 25/12*x - 25/12*y + 1

5/64*x^4*y^4 - 65/96*x^4*y^3 - 65/96*x^3*y^4 + 115/64*x^4*y^2 + 841/144*x^3*y^3 + 115/64*x^2*y^4 - 45/32*x^4*y - 1487/96*x^3*y^2 - 1487/96*x^2*y^3 - 45/32*x*y^4 + 1/24*x^4 + 1763/144*x^3*y + 2645/64*x^2*y^2 + 1763/144*x*y^3 + 1/24*y^4 - 5/12*x^3 - 3209/96*x^2*y - 3209/96*x*y^2 - 5/12*y^3 + 35/24*x^2 + 4129/144*x*y + 35/24*y^2 - 25/12*x - 25/12*y + 1

-5/96*x^4*y^4 + 25/72*x^4*y^3 + 25/72*x^3*y^4 - 55/96*x^4*y^2 - 157/72*x^3*y^3 - 55/96*x^2*y^4 + 5/72*x^4*y + 215/72*x^3*y^2 + 215/72*x^2*y^3 + 5/72*x*y^4 + 1/24*x^4 + 55/72*x^3*y - 101/96*x^2*y^2 + 55/72*x*y^3 + 1/24*y^4 - 5/12*x^3 - 515/72*x^2*y - 515/72*x*y^2 - 5/12*y^3 + 35/24*x^2 + 893/72

-5/96*x^4*y^4 + 55/144*x^4*y^3 + 55/144*x^3*y^4 - 25/32*x^4*y^2 - 67/24*x^3*y^3 - 85/96*x^2*y^4 + 35/144*x^4*y + 821/144*x^3*y^2 + 935/144*x^2*y^3 + 55/72*x*y^4 + 1/4*x^4 - 41/24*x^3*y - 433/32*x^2*y^2 - 35/6*x*y^3 - 1/6*y^4 - 2*x^3 + 667/144*x^2*y + 965/72*x*y^2 + 3/2*y^3 + 19/4*x^2 - 89/12*x*y - 13/3*y^2 - 3*x + 4*y

5/144*x^4*y^4 - 5/18*x^4*y^3 - 5/24*x^3*y^4 + 25/36*x^4*y^2 + 61/36*x^3*y^3 + 5/18*x^2*y^4 - 95/144*x^4*y - 13/3*x^3*y^2 - 169/72*x^2*y^3 + 5/48*x*y^4 + 1/4*x^4 + 319/72*x^3*y + 899/144*x^2*y^2 - 59/72*x*y^3 - 1/6*y^4 - 2*x^3 - 1075/144*x^2*y + 35/16*x*y^2 + 3/2*y^3 + 19/4*x^2 - 5/9*x*y - 13/3*y^2 - 3*x + 4*y

5/144*x^4*y^4 - 5/18*x^4*y^3 - 5/18*x^3*y^4 + 85/144*x^4*y^2 + 20/9*x^3*y^3 + 85/144*x^2*y^4 - 5/36*x^4*y - 85/18*x^3*y^2 - 85/18*x^2*y^3 - 5/36*x*y^4 - 1/6*x^4 + 43/36*x^3*y + 1409/144*x^2*y^2 + 37/36*x*y^3 - 1/6*y^4 + 7/6*x^3 - 67/36*x^2*y - 31/36*x*y^2 + 3/2*y^3 - 7/3*x^2 - 31/9*x*y - 13/3*y^2 + 4/3*x + 4*y

5/64*x^4*y^4 - 175/288*x^4*y^3 - 55/96*x^3*y^4 + 265/1

-5/96*x^4*y^4 + 65/144*x^4*y^3 + 55/144*x^3*y^4 - 125/96*x^4*y^2 - 239/72*x^3*y^3 - 25/32*x^2*y^4 + 95/72*x^4*y + 1391/144*x^3*y^2 + 979/144*x^2*y^3 + 35/144*x*y^4 - 1/6*x^4 - 89/9*x^3*y - 1931/96*x^2*y^2 - 157/72*x*y^3 + 1/4*y^4 + 7/6*x^3 + 1525/72*x^2*y + 1075/144*x*y^2 - 2*y^3 - 7/3*x^2 - 355/36*x*y + 19/4*y^2 + 4/3*x - 3*y

-5/576*x^4*y^4 + 35/288*x^4*y^3 + 25/288*x^3*y^4 - 295/576*x^4*y^2 - 155/144*x^3*y^3 - 115/576*x^2*y^4 + 175/288*x^4*y + 1235/288*x^3*y^2 + 685/288*x^2*y^3 - 25/288*x*y^4 + 1/24*x^4 - 727/144*x^3*y - 5489/576*x^2*y^2 + 47/144*x*y^3 + 1/4*y^4 - 1/4*x^3 + 3353/288*x^2*y + 421/288*x*y^2 - 2*y^3 + 11/24*x^2 - 641/144*x*y + 19/4*y^2 - 1/4*x - 3*y

-5/96*x^4*y^4 + 55/144*x^4*y^3 + 55/144*x^3*y^4 - 85/96*x^4*y^2 - 197/72*x^3*y^3 - 25/32*x^2*y^4 + 55/72*x^4*y + 887/144*x^3*y^2 + 773/144*x^2*y^3 + 35/144*x*y^4 - 1/6*x^4 - 47/9*x^3*y - 369/32*x^2*y^2 - 91/72*x*y^3 + 1/4*y^4 + 7/6*x^3 + 701/72*x^2*y + 283/144*x*y^2 - 2*y^3 - 7/3*x^2 - 91/36*x*y + 19/4*y^2 + 4/3*x - 3*y

-5

5/64*x^4*y^4 - 65/96*x^4*y^3 - 55/96*x^3*y^4 + 115/64*x^4*y^2 + 719/144*x^3*y^3 + 75/64*x^2*y^4 - 45/32*x^4*y - 1273/96*x^3*y^2 - 333/32*x^2*y^3 - 15/32*x*y^4 + 1/24*x^4 + 1477/144*x^3*y + 1789/64*x^2*y^2 + 673/144*x*y^3 - 1/6*y^4 - 1/4*x^3 - 2065/96*x^2*y - 1315/96*x*y^2 + 7/6*y^3 + 11/24*x^2 + 1595/144*x*y - 7/3*y^2 - 1/4*x + 4/3*y

-55/576*x^4*y^4 + 235/288*x^4*y^3 + 205/288*x^3*y^4 - 1265/576*x^4*y^2 - 295/48*x^3*y^3 - 905/576*x^2*y^4 + 545/288*x^4*y + 4835/288*x^3*y^2 + 3965/288*x^2*y^3 + 275/288*x*y^4 - 1/6*x^4 - 713/48*x^3*y - 22111/576*x^2*y^2 - 413/48*x*y^3 + 1/24*y^4 + 3/2*x^3 + 10159/288*x^2*y + 7141/288*x*y^2 - 1/4*y^3 - 13/3*x^2 - 1159/48*x*y + 11/24*y^2 + 4*x - 1/4*y

-5/576*x^4*y^4 + 5/32*x^4*y^3 + 35/288*x^3*y^4 - 415/576*x^4*y^2 - 239/144*x^3*y^3 - 235/576*x^2*y^4 + 95/96*x^4*y + 1945/288*x^3*y^2 + 473/96*x^2*y^3 + 85/288*x*y^4 - 1/6*x^4 - 1255/144*x^3*y - 10721/576*x^2*y^2 - 517/144*x*y^3 + 1/24*y^4 + 3/2*x^3 + 2225/96*x^2*y + 3911/288*x*y^2 - 1/4*y^3 - 13/3*x^2 - 248

In [3]:
def fuzzyPolynomial(perm,n, k, Ring):
    x, y = Ring.gens()
    p = Permutation(perm, check_input = False)
    f = factorial(n-k)/binomial(n,k)*sum(binomial(x,j)*binomial(n-1-x,k-1-j)*binomial(y,p[j])*binomial(n-1-y,k-1-p[j]) for j in range(k))
    return Ring(f)


In [33]:
R.<x,y> = PolynomialRing(QQ)
ls = [[[1], [0, 1, 2, 3, 4, 5]], [[1], [1, 2, 3, 4, 5, 0]], [[1], [2, 3, 4, 5, 0, 1]], [[1], [3, 4, 5, 0, 1, 2]], [[1], [4, 5, 0, 1, 2, 3]], [[1], [5, 0, 1, 2, 3, 4]]]
polylist = [0]*6
for i , term in enumerate(ls):
    #print(term[1][:2])
    polylist[i] = fuzzyPolynomialFixed(term[1][:],6,6, R)

In [34]:
R.<x,y> = PolynomialRing(QQ)
monomial_basis = monomials([x,y], [6,6])
b = vector(QQ, [1]+[0]*(len(monomial_basis)-1))

coeff_vectors = [vector(QQ, [f.monomial_coefficient(m) for m in monomial_basis]) for f in polylist]

A = matrix(coeff_vectors)
try:

    soln = A.solve_left(b)
    print(soln)
except ValueError:
    print("uh oh")

(1, 1, 1, 1, 1, 1)


In [87]:
for n in range(3,19):
    print(len([*possibleLengths(n,6)]))

3
12
28
49
72
96
119
139
156
169
179
185
189
191
192
192


In [95]:
print(genPermMatrix(j_perm([1,2,3,4,5]),5))

[1 0 0 0 0]
[0 1 0 0 0]
[0 0 1 0 0]
[0 0 0 1 0]
[0 0 0 0 1]


In [20]:
def fuzzyPolynomialFixed(perm,n, k, Ring):
    x, y = Ring.gens()
    p = Permutation(perm, check_input = False)
    f = factorial(n-k)/binomial(n,k)*sum(binomial(x,p[j])*binomial(n-1-x,k-1-p[j])*binomial(y,j)*binomial(n-1-y,k-1-j) for j in range(len(perm)))
    return Ring(f)

In [21]:
R.<x,y> = PolynomialRing(QQ)
ls = [[[1], [0, 1, 2, 3, 4, 5]], [[1], [1, 2, 3, 4, 5, 0]], [[1], [2, 3, 4, 5, 0, 1]], [[1], [3, 4, 5, 0, 1, 2]], [[1], [4, 5, 0, 1, 2, 3]], [[1], [5, 0, 1, 2, 3, 4]]]
polylist = [0]*6
for i , term in enumerate(ls):
    #print(term[1][:2])
    polylist[i] = fuzzyPolynomialFixed(term[1][:2],6,6, R)

In [18]:
for poly in polylist:
    print(poly)
    print(Q(poly))
    print()

7/400*x^5*y^5 - 7/32*x^5*y^4 - 7/32*x^4*y^5 + 343/360*x^5*y^3 + 791/288*x^4*y^4 + 343/360*x^3*y^5 - 161/96*x^5*y^2 - 385/32*x^4*y^3 - 385/32*x^3*y^4 - 161/96*x^2*y^5 + 439/450*x^5*y + 6145/288*x^4*y^2 + 2549/48*x^3*y^3 + 6145/288*x^2*y^4 + 439/450*x*y^5 - 1/120*x^5 - 301/24*x^4*y - 3045/32*x^3*y^2 - 3045/32*x^2*y^3 - 301/24*x*y^4 - 1/120*y^5 + 1/8*x^4 + 20447/360*x^3*y + 49847/288*x^2*y^2 + 20447/360*x*y^3 + 1/8*y^4 - 17/24*x^3 - 847/8*x^2*y - 847/8*x*y^2 - 17/24*y^3 + 15/8*x^2 + 122269/1800*x*y + 15/8*y^2 - 137/60*x - 137/60*y + 1
-161/96*xbar^5*ybar^2 + 439/450*xbar^5*ybar + 6145/288*xbar^4*ybar^2 - 1/120*xbar^5 - 301/24*xbar^4*ybar - 3045/32*xbar^3*ybar^2 + 1/8*xbar^4 + 20447/360*xbar^3*ybar + 49847/288*xbar^2*ybar^2 - 17/24*xbar^3 - 847/8*xbar^2*ybar - 847/8*xbar*ybar^2 + 15/8*xbar^2 + 122269/1800*xbar*ybar + 15/8*ybar^2 - 137/60*xbar - 137/60*ybar + 1

-211/14400*x^5*y^5 + 137/720*x^5*y^4 + 169/960*x^4*y^5 - 2513/2880*x^5*y^3 - 661/288*x^4*y^4 - 701/960*x^3*y^5 + 1189/720*x^5*y^2 

In [22]:
for poly in polylist:
    print(poly)
    print(Q(poly))
    print()

13/7200*x^5*y^5 - 73/2880*x^5*y^4 - 73/2880*x^4*y^5 + 31/240*x^5*y^3 + 205/576*x^4*y^4 + 31/240*x^3*y^5 - 163/576*x^5*y^2 - 1045/576*x^4*y^3 - 1045/576*x^3*y^4 - 163/576*x^2*y^5 + 1637/7200*x^5*y + 2291/576*x^4*y^2 + 2665/288*x^3*y^3 + 2291/576*x^2*y^4 + 1637/7200*x*y^5 - 1/120*x^5 - 1537/480*x^4*y - 11699/576*x^3*y^2 - 11699/576*x^2*y^3 - 1537/480*x*y^4 - 1/120*y^5 + 1/8*x^4 + 23629/1440*x^3*y + 25741/576*x^2*y^2 + 23629/1440*x*y^3 + 1/8*y^4 - 17/24*x^3 - 3491/96*x^2*y - 3491/96*x*y^2 - 17/24*y^3 + 15/8*x^2 + 108769/3600*x*y + 15/8*y^2 - 137/60*x - 137/60*y + 1
-163/576*xbar^5*ybar^2 + 1637/7200*xbar^5*ybar + 2291/576*xbar^4*ybar^2 - 1/120*xbar^5 - 1537/480*xbar^4*ybar - 11699/576*xbar^3*ybar^2 + 1/8*xbar^4 + 23629/1440*xbar^3*ybar + 25741/576*xbar^2*ybar^2 - 17/24*xbar^3 - 3491/96*xbar^2*ybar - 3491/96*xbar*ybar^2 + 15/8*xbar^2 + 108769/3600*xbar*ybar + 15/8*ybar^2 - 137/60*xbar - 137/60*ybar + 1

-11/2880*x^5*y^5 + 31/576*x^5*y^4 + 1/20*x^4*y^5 - 53/192*x^5*y^3 - 203/288*x^4*y^4 - 6

In [6]:
for sigma in Permutations(2):
    print(combination_of_permutations([1],[sigma]))
    print(combination_of_permutations([1],[sigma]).genFuzzyMatrix(4))
    print()

1*(12) 
[  3   2   1   0]
[  2 5/3 4/3   1]
[  1 4/3 5/3   2]
[  0   1   2   3]

1*(21) 
[  0   1   2   3]
[  1 4/3 5/3   2]
[  2 5/3 4/3   1]
[  3   2   1   0]



In [13]:
for s1 in Permutations(4):
    print(s1)
    m1 = combination_of_permutations([1],[s1]).genFuzzyMatrix(4).list()
    for s2 in Permutations(4):
        if str(s2) < str(s1) or str(s2) == str(s1):
            continue
        m2 = combination_of_permutations([1],[s2]).genFuzzyMatrix(4).list()
        for s3 in Permutations(4):
            if str(s3) < str(s2) or str(s3) == str(s2):
                continue
            m3 = combination_of_permutations([1],[s3]).genFuzzyMatrix(4).list()
            for s4 in Permutations(3):
                m4 = combination_of_permutations([1],[s4]).genFuzzyMatrix(4).list()
                m = Matrix(QQ,[m1,m2,m3]).transpose()
                try:
                    soln = m.solve_right(vector(QQ,-1*m4))
                    print(soln)
                except ValueError:
                    pass

[1, 2, 3, 4]
[1, 2, 4, 3]
[1, 3, 2, 4]
[1, 3, 4, 2]
[1, 4, 2, 3]
[1, 4, 3, 2]
[2, 1, 3, 4]
[2, 1, 4, 3]
[2, 3, 1, 4]
[2, 3, 4, 1]
[2, 4, 1, 3]
[2, 4, 3, 1]
[3, 1, 2, 4]
[3, 1, 4, 2]
[3, 2, 1, 4]
[3, 2, 4, 1]
[3, 4, 1, 2]
[3, 4, 2, 1]
[4, 1, 2, 3]
[4, 1, 3, 2]
[4, 2, 1, 3]
[4, 2, 3, 1]
[4, 3, 1, 2]
[4, 3, 2, 1]


In [14]:
for s1 in Permutations(5):
    print(s1)
    m1 = combination_of_permutations([1],[s1]).genFuzzyMatrix(5).list()
    for s2 in Permutations(5):
        if str(s2) < str(s1) or str(s2) == str(s1):
            continue
        m2 = combination_of_permutations([1],[s2]).genFuzzyMatrix(5).list()
        for s3 in [j_perm([2,4,1,3]),j_perm([3,1,4,2])]:
            m3 = combination_of_permutations([1],[s3]).genFuzzyMatrix(5).list()
            m4 = combination_of_permutations([1],[j_perm([1,2])]).genFuzzyMatrix(5).list()
            m = Matrix(QQ,[m1,m2,m3]).transpose()
            try:
                soln = m.solve_right(vector(QQ,-1*m4))
                print(soln)
            except ValueError:
                pass

[1, 2, 3, 4, 5]
[1, 2, 3, 5, 4]
[1, 2, 4, 3, 5]
[1, 2, 4, 5, 3]
[1, 2, 5, 3, 4]
[1, 2, 5, 4, 3]
[1, 3, 2, 4, 5]
[1, 3, 2, 5, 4]
[1, 3, 4, 2, 5]
[1, 3, 4, 5, 2]
[1, 3, 5, 2, 4]
[1, 3, 5, 4, 2]
[1, 4, 2, 3, 5]
[1, 4, 2, 5, 3]
[1, 4, 3, 2, 5]
[1, 4, 3, 5, 2]
[1, 4, 5, 2, 3]
[1, 4, 5, 3, 2]
[1, 5, 2, 3, 4]
[1, 5, 2, 4, 3]
[1, 5, 3, 2, 4]
[1, 5, 3, 4, 2]
[1, 5, 4, 2, 3]
[1, 5, 4, 3, 2]
[2, 1, 3, 4, 5]
[2, 1, 3, 5, 4]
[2, 1, 4, 3, 5]
[2, 1, 4, 5, 3]
[2, 1, 5, 3, 4]
[2, 1, 5, 4, 3]
[2, 3, 1, 4, 5]
[2, 3, 1, 5, 4]
[2, 3, 4, 1, 5]
[2, 3, 4, 5, 1]
[2, 3, 5, 1, 4]
[2, 3, 5, 4, 1]
[2, 4, 1, 3, 5]
[2, 4, 1, 5, 3]
[2, 4, 3, 1, 5]
[2, 4, 3, 5, 1]
[2, 4, 5, 1, 3]
[2, 4, 5, 3, 1]
[2, 5, 1, 3, 4]
[2, 5, 1, 4, 3]
[2, 5, 3, 1, 4]
[2, 5, 3, 4, 1]
[2, 5, 4, 1, 3]
[2, 5, 4, 3, 1]
[3, 1, 2, 4, 5]
[3, 1, 2, 5, 4]
[3, 1, 4, 2, 5]
[3, 1, 4, 5, 2]
[3, 1, 5, 2, 4]
[3, 1, 5, 4, 2]
[3, 2, 1, 4, 5]
[3, 2, 1, 5, 4]
[3, 2, 4, 1, 5]
[3, 2, 4, 5, 1]
[3, 2, 5, 1, 4]
[3, 2, 5, 4, 1]
[3, 4, 1, 2, 5]
[3, 4, 1, 5, 2]
[3, 4, 2

In [15]:
for s1 in Permutations(4):
    print(s1)
    m1 = combination_of_permutations([1],[s1]).genFuzzyMatrix(4).list()
    for s2 in Permutations(4):
        if str(s2) < str(s1) or str(s2) == str(s1):
            continue
        m2 = combination_of_permutations([1],[s2]).genFuzzyMatrix(4).list()
        for s3 in [j_perm([2,1,3]),j_perm([3,1,2])]:
            m3 = combination_of_permutations([1],[s3]).genFuzzyMatrix(4).list()
            m4 = combination_of_permutations([1],[j_perm([1,2])]).genFuzzyMatrix(4).list()
            m = Matrix(QQ,[m1,m2,m3]).transpose()
            try:
                soln = m.solve_right(vector(QQ,-1*m4))
                print(soln)
            except ValueError:
                pass

[1, 2, 3, 4]
[1, 2, 4, 3]
[1, 3, 2, 4]
[1, 3, 4, 2]
[1, 4, 2, 3]
[1, 4, 3, 2]
[2, 1, 3, 4]
[2, 1, 4, 3]
[2, 3, 1, 4]
[2, 3, 4, 1]
[2, 4, 1, 3]
[2, 4, 3, 1]
[3, 1, 2, 4]
[3, 1, 4, 2]
[3, 2, 1, 4]
[3, 2, 4, 1]
[3, 4, 1, 2]
[3, 4, 2, 1]
[4, 1, 2, 3]
[4, 1, 3, 2]
[4, 2, 1, 3]
[4, 2, 3, 1]
[4, 3, 1, 2]
[4, 3, 2, 1]


In [16]:
for s1 in Permutations(3):
    print(s1)
    m1 = combination_of_permutations([1],[s1]).genFuzzyMatrix(3).list()
    for s2 in Permutations(3):
        if str(s2) < str(s1) or str(s2) == str(s1):
            continue
        m2 = combination_of_permutations([1],[s2]).genFuzzyMatrix(3).list()
        for s3 in Permutations(2):
            if str(s3) < str(s2) or str(s3) == str(s2):
                continue
            m3 = combination_of_permutations([1],[s3]).genFuzzyMatrix(3).list()
            m4 = combination_of_permutations([1],[j_perm([2,1])]).genFuzzyMatrix(3).list()
            m = Matrix(QQ,[m1,m2,m3]).transpose()
            try:
                soln = m.solve_right(vector(QQ,-1*m4))
                print(soln)
            except ValueError:
                pass
            
            try:
                soln = m.solve_right(vector(QQ,m4))
                print(soln)
            except ValueError:
                pass

[1, 2, 3]
(0, 0, 1)
(0, 0, 1)
[1, 3, 2]
(0, 0, 1)
[2, 1, 3]
[2, 3, 1]
[3, 1, 2]
[3, 2, 1]


In [25]:
n = 6
flat12 = genPermMatrix([0,1],6).list()
for s1 in Permutations(n):
    print("s1:"+str(s1))
    m1 = combination_of_permutations([1],[s1]).genFuzzyMatrix(n).list()
    for s2 in Permutations(n):
        if str(s2) < str(s1) or str(s2) == str(s1):
            continue
        print("s2:"+str(s2))
        m2 = combination_of_permutations([1],[s2]).genFuzzyMatrix(n).list()
        for s3 in Permutations(n):
            if str(s3) < str(s2) or str(s3) == str(s2):
                continue
            m3 = combination_of_permutations([1],[s3]).genFuzzyMatrix(n).list()
            for s4 in Permutations(n-1):
                m4 = combination_of_permutations([1],[s4]).genFuzzyMatrix(n).list()
            m = Matrix(QQ,[m1,m2,m3,m4]).transpose()
            try:
                soln = m.solve_right(vector(QQ,-1*flat12))
                print(soln)
            except ValueError:
                pass

s1:[1, 2, 3, 4, 5, 6]
s2:[1, 2, 3, 4, 6, 5]


KeyboardInterrupt: 

In [3]:
print(combination_of_permutations([1],[[1,2]]).genFuzzyMatrix(3))

[4/3 2/3   0]
[2/3 2/3 2/3]
[  0 2/3 4/3]


In [8]:
sum([len([*possibleLengths(n,6)]) for n in range(3,19)])

1971

The vanishing cover xi + 3/2nu is the smallest vanishign cover. Hence our search is wrong because this little guy is so small. Double check our $n$.

In [13]:
print(xi.genFuzzyMatrix(3))
print(nu.genFuzzyMatrix(3))

diff = xi + 3/2*nu
print(diff.isVanishing(3))
for m in diff.genFuzzyMatrixList(3):
    print(m)
    
print(diff.get_terms())

[-2 -2 -2]
[-2 -2 -2]
[-2 -2 -2]
[4/3 4/3 4/3]
[4/3 4/3 4/3]
[4/3 4/3 4/3]
True
[2 0 0]
[0 2 0]
[0 0 2]
[ 0  0 -2]
[ 0 -2  0]
[-2  0  0]
[-2 -1  0]
[-1 -1 -1]
[ 0 -1 -2]
[0 1 2]
[1 1 1]
[2 1 0]
[[0, 1, 2], [2, 1, 0], [0, 1], [1, 0]]


In [6]:
cover_list = []
for lengths in possibleLengths(3,4):
    cover_list.append(searchRecur(lengths))


2026-06-07 11:31:20.255386
2026-06-07 11:31:20.256291
Examining [0]
Found cover with perms: [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 0]]
Examining [1]
Found cover with perms: [[1, 2, 0], [2, 0, 1], [2, 1, 0], [0, 1]]
2026-06-07 11:31:20.551986
Examining [0]
Found infFam with perms: [[0, 1, 2], [2, 1, 0], [0, 1], [1, 0]]
Examining [1]


In [15]:
print(cover_list)
for thing in cover_list:
    print(thing[0])
    
rho1 = combination_of_permutations([2,2,2,3],[[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 0]])
print(rho1.genFuzzyMatrix(3))
print(nu.genFuzzyMatrix(3))

diff = rho1 - 3*nu
print(diff.isVanishing(3))
print(combination_of_permutations([1],[[2,4,1,3]]).genFuzzyMatrix(5))

[(['Covers of type [3, 3, 3, 3] are ruled out by length tuple alone.'], None), (['2*(123) + 2*(132) + 2*(213) + 3*(21) '], None), (['infFam:-1*(123) + 1*(321) + 3/2*(12) +t*(1*(123) -1*(321) -3/4*(12) + 3/4*(21))'], None)]
['Covers of type [3, 3, 3, 3] are ruled out by length tuple alone.']
['2*(123) + 2*(132) + 2*(213) + 3*(21) ']
['infFam:-1*(123) + 1*(321) + 3/2*(12) +t*(1*(123) -1*(321) -3/4*(12) + 3/4*(21))']
[4 4 4]
[4 4 4]
[4 4 4]
[4/3 4/3 4/3]
[4/3 4/3 4/3]
[4/3 4/3 4/3]
True
[   0 12/5  8/5    0    0]
[   0  3/5  2/5  3/5 12/5]
[ 8/5  2/5    0  2/5  8/5]
[12/5  3/5  2/5  3/5    0]
[   0    0  8/5 12/5    0]


In [3]:
sigma = Permutation([1,3,2,4])
tau = Permutation([2,4,1,3])
print((tau * sigma.inverse()).cycle_type())

[2, 2]


In [6]:
print(genPermMatrixByRow(sigma, 7, 5,4))
print()
print(genPermMatrixByRow(sigma, 7, 7,4))

[     0  240/7  288/7  216/7   96/7      0      0]
[     0  120/7  144/7  120/7   96/7  120/7  240/7]
[     0   48/7 384/35   72/5   96/5  192/7  288/7]
[     0   12/7 288/35 594/35 888/35  216/7  216/7]
[     0      0 288/35   96/5  192/7  192/7   96/7]

[     0  240/7  288/7  216/7   96/7      0      0]
[     0  120/7  144/7  120/7   96/7  120/7  240/7]
[     0   48/7 384/35   72/5   96/5  192/7  288/7]
[     0   12/7 288/35 594/35 888/35  216/7  216/7]
[     0      0 288/35   96/5  192/7  192/7   96/7]
[     0      0   48/7  108/7  144/7  120/7      0]
[     0      0      0      0      0      0      0]


In [7]:
for n in range(6,19):
    for tup in [*possibleLengths(n,6)]:
        if tup[-1] == 2 and tup[-2] == 2:
            print(tup)

[6, 6, 6, 5, 2, 2]
[6, 6, 6, 4, 2, 2]
[6, 6, 5, 5, 2, 2]
[6, 6, 5, 4, 2, 2]
[6, 6, 5, 3, 2, 2]
[7, 7, 7, 5, 2, 2]
[7, 7, 6, 6, 2, 2]
[7, 7, 6, 5, 2, 2]
[7, 7, 6, 4, 2, 2]
[8, 8, 7, 6, 2, 2]
[8, 8, 7, 5, 2, 2]
[9, 9, 8, 6, 2, 2]


In [20]:
searchRecur([9, 9, 8, 6, 2, 2])

2026-06-18 14:40:51.591296
Examining [0]
The 10th case: [[0, 1, 2, 3, 4, 5, 7], [0, 1, 2, 3, 4, 5, 8], [0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 20th case: [[0, 1, 2, 3, 4, 5, 7, 6], [0, 1, 2, 3, 4, 5, 7, 8], [0, 1, 2, 3, 4, 6, 5, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 30th case: [[0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 4, 5, 8], [0, 1, 2, 3, 4, 7, 5], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 40th case: [[0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 6], [0, 1, 2, 3, 4, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 50th case: [[0, 1, 2, 3, 4, 6, 5], [0, 1, 2, 3, 4, 6, 7], [0, 1, 2, 3, 4, 5, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 60th case: [[0, 1, 2, 3, 4, 6, 5, 7], [0, 1, 2, 3, 4, 6, 5, 8], [0, 1, 2, 3, 4, 6, 7, 5], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 70th case: [[0, 1, 2, 3, 4, 6, 8, 5], [0, 1, 2, 3, 4, 6, 8, 7], [0, 1, 2, 3, 4, 6, 7, 5], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 80th case: [[0, 1, 2, 3, 4, 6, 7], [0, 1, 2, 3, 4, 6, 8], [0, 1, 2, 3, 4, 7, 6], [0, 1, 2,

The 670th case: [[0, 1, 2, 3, 4, 5, 7], [0, 1, 2, 3, 4, 5, 8], [0, 1, 2, 3, 6, 5, 4], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 680th case: [[0, 1, 2, 3, 4, 5, 7, 6], [0, 1, 2, 3, 4, 5, 7, 8], [0, 1, 2, 3, 6, 7, 4, 5], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 690th case: [[0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 7], [0, 1, 2, 3, 6, 5], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 700th case: [[0, 1, 2, 3, 4, 6, 5], [0, 1, 2, 3, 4, 6, 8], [0, 1, 2, 3, 6, 4, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 710th case: [[0, 1, 2, 3, 4, 6, 5], [0, 1, 2, 3, 4, 6, 7], [0, 1, 2, 3, 6, 5, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 720th case: [[0, 1, 2, 3, 4, 6, 5, 7], [0, 1, 2, 3, 4, 6, 5, 8], [0, 1, 2, 3, 6, 7, 5, 4], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 730th case: [[0, 1, 2, 3, 4, 6, 8, 5], [0, 1, 2, 3, 4, 6, 8, 7], [0, 1, 2, 3, 6, 7, 5, 4], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]
The 740th case: [[0, 1, 2, 3, 4, 7, 5], [0, 1, 2, 3, 4, 7, 6], [0, 1, 2, 3, 6, 4, 7], [0, 1, 2, 3, 4, 5], [0, 1], [1, 0]]


KeyboardInterrupt: 

In [22]:
genPermMatrix([0,1],6)

[   40    32    24    16     8     0]
[   32 136/5 112/5  88/5  64/5     8]
[   24 112/5 104/5  96/5  88/5    16]
[   16  88/5  96/5 104/5 112/5    24]
[    8  64/5  88/5 112/5 136/5    32]
[    0     8    16    24    32    40]

In [36]:
for nested_list in start_with_5_latins:
    print(combination_of_permutations.nested(nested_list).dispLatex())

\item $\rho=\perm{1} $
\item $\rho=\perm{21} +\perm{12} $
\item $\rho=\perm{321} +\perm{213} +\perm{132} $
\item $\rho=\perm{312} +\perm{231} +\perm{123} $
\item $\rho=-2(\perm{321}) +2(\perm{123}) -3(\perm{12}) $
\item $\rho=\perm{4231} +\perm{3412} +\perm{2143} +\perm{1324} $
\item $\rho=\perm{4213} +\perm{3421} +\perm{2134} +\perm{1342} $
\item $\rho=\perm{4321} +\perm{3412} +\perm{2143} +\perm{1234} $
\item $\rho=\perm{4321} +\perm{3412} +\perm{2134} +\perm{1243} $
\item $\rho=\perm{4312} +\perm{3421} +\perm{2143} +\perm{1234} $
\item $\rho=\perm{4321} +\perm{3214} +\perm{2143} +\perm{1432} $
\item $\rho=\perm{4312} +\perm{3241} +\perm{2134} +\perm{1423} $
\item $\rho=\perm{4321} +\perm{3142} +\perm{2413} +\perm{1234} $
\item $\rho=\perm{4312} +\perm{3124} +\perm{2431} +\perm{1243} $
\item $\rho=\perm{4231} +\perm{3124} +\perm{2413} +\perm{1342} $
\item $\rho=\perm{4231} +\perm{3142} +\perm{2314} +\perm{1423} $
\item $\rho=\perm{4213} +\perm{3124} +\perm{2341} +\perm{1432} $
\item 